# K-Merge Merge Lab: where LoRA merging breaks, and a new way to fix it

**One notebook, no wiring needed.** It builds its own data, trains its own adapters, runs every merge method, and prints a summary you can send back.

| Step | What happens |
|---|---|
| 1 | Builds grammar-correction data for **English, German, French, Hindi** (templated, several error types, 20% already-correct sentences, no sentence shared between train/val/test) |
| 2 | Trains one **LoRA adapter per language per seed** (rank 8, `q_proj`+`v_proj`, Qwen2.5-Instruct) |
| 3 | Puts every pair of adapters into **one rank-8 slot** (the size of one K-Merge slot) with **22 merge methods** (prior work + GRAM) and **3 no-merge K-Share variants** |
| 4 | Measures *why* a merge fails, not just the score: under-correction (dilution), language leakage (interference), over-correction, imbalance, per-layer conflict |
| 5 | Tests whether a **data-free distortion score** predicts merge damage better than **cosine similarity** (K-Merge's rule) |
| 6 | Re-runs **online K-Merge** (adapters arrive one by one, K=2 slots) over **all arrival orders**: merging rules × 4 partner-selection policies, and K-Share with **per-layer slot assignment** |
| 7 | **Slot capacity:** forces 3 and 4 tasks into one slot to see where merging, `split` and K-Share break, then reports confidence intervals and significance tests for every comparison |

### How to run on Kaggle
1. *Settings* → **Accelerator: GPU T4 x2** (or P100), **Internet: ON** (to download Qwen from Hugging Face).
2. First run with `RUN_MODE = "SMOKE"` (about 3 min on a T4): it only checks that everything works.
3. Then set `RUN_MODE = "QUICK"` (seed 0) and finally `"FULL"` (seeds 0-2).
   For long runs use **Save Version → Save & Run All (Commit)** so it keeps running if your browser closes.
4. Everything is cached in `/kaggle/working/kmerge_lab`. **Resume / reuse:** right panel → *Add Input* → *Your Work* → this notebook, then run again: finished work from that earlier version (same model and settings) is copied in and skipped, so FULL reuses all of QUICK's seed-0 work.
5. **Send back:** the text printed under `SEND THIS BACK` (also saved as `RESULTS_SUMMARY.txt`) and the file `results.zip` from the Output tab. Tables [11]-[12] give 95% confidence intervals and Holm-corrected p-values (paired bootstrap over test sentences, plus McNemar's test) for the paper.

To test the 1.5B model (28 layers, where your layer 17-21 finding came from), set `MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"` and use QUICK first.

## Part A: the merge-method landscape (what research has tried)

| Family | Methods (year) | Core idea | Pros | Cons / where it lags | Data needed | In this notebook |
|---|---|---|---|---|---|---|
| **A. Averaging** | Linear/AdapterSoup, Model Soups, **K-Merge Eq. 8** ([2025/26](https://arxiv.org/abs/2510.13537)), SLERP | average the adapters (history-weighted) | cheap, stable, order-invariant | **dilution**: where tasks don't overlap, each task keeps only 1/n of its update | none | `kmerge`, `delta_avg`, `slerp` |
| **A'. Scaling** | Task Arithmetic ([2023](https://arxiv.org/abs/2212.04089)), LiNeS (2025), LARV (2026) | add task vectors, scale by λ (global or per layer) | fixes dilution | one λ needs labeled validation data; bigger λ = more interference | labeled val | `task_arith` |
| **B. Sparsify / sign** | TIES ([2023](https://arxiv.org/abs/2306.01708)), DARE ([2024](https://arxiv.org/abs/2311.03099)), DELLA, Breadcrumbs, MagMax (2024), Consensus/TALL masks, PCB (2024) | drop small or random entries, vote on signs | fewer element collisions | element/sign view is meaningless for LoRA factors; dense result must be squeezed back to rank r (large loss) | val for k/p | `ties`, `dare`, `magmax` |
| **C. Subspace / SVD** | KnOTS (2025), TSV-M ([2025](https://arxiv.org/abs/2412.00081)), Iso-C/Iso-CTS ([2025](https://arxiv.org/abs/2502.04959)), Core Space ([2025](https://arxiv.org/abs/2509.17786)), CtM ([2026](https://arxiv.org/abs/2606.03723)), LoRA-LEGO, RobustMerge | align / decorrelate singular directions, flatten spectra | less cross-task interference | importance = weight size (Frobenius), not what the task *uses*; mostly offline multi-task | none / val | `knots_ties`, `tsv`, `iso_c` |
| **D. Functional (regression)** | Fisher (2022), RegMean ([2023](https://arxiv.org/abs/2212.09849)), LoRM ([2024](https://arxiv.org/abs/2410.17961)), WUDI ([2025](https://arxiv.org/abs/2503.08099)) | minimise the change in each layer's *output* | targets the real problem | RegMean/Fisher/LoRM need activations; WUDI is data-free but iterative and not rank-budget aware | activations / none | `regmean_cf`, `selfgram_cf` |
| **E. Learned coefficients** | AdaMerging (2024), LoraHub (2024), Surgery | optimise merge weights on data | strong | needs data + optimisation on device | (un)labeled data | (out of scope) |
| **F. Keep separate / route** | MoLE, Twin-Merging, EMR-Merging, Arrow routing | don't fully merge; route per input | avoids interference | extra storage/router: breaks the K-slot budget | varies | (out of scope) |
| **G. Continual / online** | OPCM ([2025](https://arxiv.org/abs/2501.09522)), MagMax, K-Merge / K-Merge++, D²C ([2026](https://arxiv.org/abs/2601.17441)) | merge as tasks arrive | realistic on-device | partner chosen by similarity (or needs data); sensitive to arrival order | none / few examples | 4 selection policies |
| **H. Layer / structure** | Layer swapping ([2025](https://arxiv.org/abs/2410.01335)), modular cross-lingual merging ([2025](https://arxiv.org/abs/2505.18356)), your Analysis-4 layer exclusion | language info and task info live in different layers | explains *where* conflict is | must know which layers | none | `layer_sel`, `ocp`, `casc_soft` |

Surveys and benchmarks: [Model Merging in LLMs, MLLMs, and Beyond (ACM CSUR 2026)](https://arxiv.org/abs/2408.07666), [MergeBench (2025)](https://arxiv.org/abs/2505.10833).

## Part B: the common weakness (where every family lags)

Each family fixes one symptom. They share one root cause: **they judge an adapter by its weights, not by what it does to the inputs it actually sees**, and **they ignore the fixed rank budget until the very end** (merge, then squeeze back to rank r). That produces measurable failure modes:

| # | Failure mode | Symptom you already saw | Methods that suffer | How this notebook measures it |
|---|---|---|---|---|
| F1 | **Dilution** | merged model stops correcting and copies the input | averaging, K-Merge, TIES/DARE disjoint mean | `copy` rate on erroneous inputs |
| F2 | **Functional interference** | wrong-language tokens ("He" inside French output) | sums, task arithmetic, TSV | `leak` rate, per-layer functional distortion |
| F3 | **Rank-truncation bias** | after squeezing to rank r one language wins, the other loses | every merge-then-SVD method | per-task imbalance, truncation loss |
| F4 | **Norm dominance** | the larger adapter (e.g. Hindi) always gains, the other drops | averaging, sign election, SVD truncation | correlation of norm ratio with imbalance |
| F5 | **Similarity ≠ compatibility** | EN-HI most similar yet EN loses; results depend on arrival order | K-Merge partner choice | Spearman(cosine, damage) vs Spearman(distortion, damage); spread across orders |
| F6 | **Weight-only proxies fail for LoRA** *(found while building this notebook)* | a merge that looks perfect on the adapter's own weights changes real outputs 6× more than averaging | WUDI-style data-free merging | `selfgram_cf` / `selfgram_ridge` vs real-input distortion |

**Why F6 happens:** WUDI's data-free trick assumes that a task vector's rows span the inputs the task sees. That is true for full fine-tuning (every gradient is *error × input*). In LoRA, $\Delta W = BA$, and the rows of $A$ are mostly its **random initialisation**, so the "footprint" describes random directions, not the task's inputs. With shared initialisation the two adapters' row spaces almost coincide, and an exact solver exploits their tiny differences with huge coefficients. **Merging LoRA functionally needs *some* information about real inputs.**

## Part C: the new method: **GRAM** (Gram-weighted, Rank-Aware Merging) + **GRAM-Select**

One objective decides both **how** to merge and **whom** to merge:

$$\min_{\operatorname{rank}(X)\le r}\;\sum_t \frac{n_t}{\sqrt{E_t}}\,\big\|(X-\Delta W_t)\,C_t^{1/2}\big\|_F^2,\qquad E_t=\|\Delta W_t\,C_t^{1/2}\|_F^2$$

* $C_t$ = covariance of the inputs task $t$ sees at that layer (from **unlabeled** prompts; never labels):
  * **GRAM-G (fully data-free):** the *base model* writes 32 short sentences from the task description alone ("a simple sentence in French"). K-Merge already assumes each adapter's task is known, because it routes inputs by task, so no extra assumption is needed.
  * **GRAM-U (tiny unlabeled buffer):** 32 real user inputs the device has already seen; they never leave the device.
* $n_t$ = how many tasks a slot already holds (K-Merge's history weight).
* $1/\sqrt{E_t}$ **balance**: for non-overlapping tasks this makes both tasks lose exactly the same *fraction* of their effect, so the larger adapter cannot dominate. Fixes F4. (Plain $1/E_t$ over-corrects towards the smaller adapter; the self-test checks this.)
* **Exact, closed form, no iterations, no hyper-parameters:**
  1. Unconstrained optimum $X^*=(\sum_t c_t\Delta W_tC_t)(\sum_t c_tC_t)^{-1}$: input directions only one task uses keep **full strength** (no dilution, F1); shared directions get a functional compromise (F2).
  2. Rank-$r$ optimum: truncate in the **whitened** space, $X=[X^*C^{1/2}]_r\,C^{-1/2}$ (Eckart–Young in the task metric). It drops what matters least **to the tasks' outputs**, not the smallest weights. Fixes F3.
* Cost: one $d\times d$ eigendecomposition per layer, milliseconds on a GPU.

**GRAM-Select (online):** when all K slots are full, consider **all pairs** among the stored slots + the new adapter, and merge the pair whose GRAM merge has the **lowest predicted distortion on the generated inputs** (data-free). K-Merge only merges the *new* adapter into its most cosine-similar slot. Example: arrival FR→DE→EN; K-Merge must merge EN into FR or DE, while GRAM-Select can merge DE+FR and keep EN alone. Fixes F5.

**What is new vs prior work (re-check before submission):** RegMean/LoRM use the same functional loss but need real activations and do not solve it exactly under the rank-r slot constraint (LoRM alternates A/B; CtM/Core Space keep rank r but optimise weight-space error). WUDI is data-free but, as F6 shows, its proxy breaks for LoRA. To our knowledge, none (i) use task-description-generated pseudo-inputs to make functional LoRA merging data-free, (ii) give the exact rank-r optimum, (iii) balance tasks by their own functional energy, or (iv) reuse the merge objective as a partner-selection score for online merging. **Ablation ladders** (one change per step):
* data-free: `selfgram_cf` (WUDI objective) → `selfgram_ridge` (+ prior, balance, rank-aware: still bad = the proxy is the problem) → `gram_iso` (ignore inputs: balanced average)
* generated inputs: `regmeanG_cf` (RegMean objective) → `gramG_nofair` (+ exact rank-r) → **`gramG`** (+ balance)
* unlabeled inputs: `regmean_cf` → `gramU_nofair` → **`gramU`**

**Honest expectation:** GRAM may not win on every pair. Either way the diagnostics show *which* failure mode dominates multilingual LoRA merging, which is publishable too.

## Part D: beyond merging: **K-Share** (task-conditional slots)

**The key observation:** K-Merge routes every input to the slot that holds its task, so **the task is known at inference**. Yet every merging method (K-Merge and the 22 above) still forces *one* set of weights to serve all tasks in a slot. That constraint is inherited from offline merging, where the task is unknown; here it is unnecessary. In effect K-Merge is online clustering of adapters in which each slot stores the *average* of its members. K-Share replaces "average" with "share".

**K-Share:** a slot stores a shared basis $U, V$ (exactly the size of one LoRA) plus, for each task, a tiny $r\times r$ core $G_t$ per module (about 0.5% of one adapter). Task $t$ runs with $U G_t V^\top$, an ordinary LoRA, so there is no slowdown, no averaging and no cross-task interference. The only loss: each task is projected onto the shared basis.
* The basis is found by HOOI (a few SVDs); each task is weighted by its own energy, so a big adapter cannot take the whole basis.
* **Online:** when a new adapter arrives and the slots are full, merge the pair (stored slots + new) that loses the least energy. Data-free.
* **Per-layer assignment:** the grouping may differ per layer (e.g. EN shares with FR everywhere except the conflict layers). Storage is unchanged; only a tiny task × layer lookup table is added.

| Failure mode | How K-Share answers it |
|---|---|
| F1 dilution | nothing is averaged: each task keeps its own coefficients |
| F2 interference | a task only ever runs with its own core |
| F3 / F4 truncation bias, norm dominance | the basis is chosen with each task weighted by its own energy |
| F5 partner choice | energy-loss score over all pairs, optionally per layer |
| F6 weight-only proxies | no input proxy is needed at all |

**Ablation ladder:** `split` (each task keeps its own top directions: the TSV-C idea, prior work) → `kshare` (shared basis) → `kshare_eq` (rank r−1 so basis + cores fit *inside* one slot). Online: `kshare` + K-Merge's policy (only the representation changes) → `global_loss` → `layer_loss` (per-layer assignment).

**Prior work to cite and differentiate:** shared basis + per-LoRA matrices for *serving* many LoRAs offline ([Compress then Serve, 2024](https://arxiv.org/abs/2407.00066)); per-task truncation offline ([TSV-C, 2025](https://arxiv.org/abs/2412.00081)); task-specific masks with known task identity for full models (EMR-Merging, [TALL masks](https://arxiv.org/abs/2405.07813), 2024); task grouping per layer *during training* ([TreeLoRA, 2025](https://www.lamda.nju.edu.cn/qianyy/paper/ICML25_TreeLoRA.pdf)). **New here, to our knowledge:** showing that K-Merge's own routing makes merging unnecessary and that storage-equivalent sharing beats merging in its setting; an online algorithm for shared slots; data-free per-layer slot assignment.

In [ ]:
# ============================================================================
# 1. SETUP AND CONFIG
# ============================================================================
import os, sys, re, json, math, time, random, itertools, difflib, subprocess, zipfile, gc, hashlib
import urllib.request
from collections import OrderedDict, Counter, defaultdict


def _internet_ok(url="https://huggingface.co", timeout=8):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False


INTERNET = _internet_ok()


def _ensure(pkg, pip_name=None):
    """Import a package; if missing and Internet is on, pip-install it. Returns True if importable."""
    try:
        __import__(pkg)
        return True
    except ImportError:
        if not INTERNET:
            return False
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], capture_output=True, text=True)
        try:
            __import__(pkg)
            return True
        except ImportError:
            print(r.stdout[-1500:], r.stderr[-1500:])
            return False


if not _ensure("peft"):
    raise RuntimeError("PRE-FLIGHT FAILED: the 'peft' library is missing and could not be installed.\n"
                       "FIX: right panel -> Settings -> Internet -> On (needs a phone-verified Kaggle account), "
                       "then Run All again.")
try:                     # an old preinstalled peft can crash with a newer transformers
    from peft import LoraConfig, get_peft_model, get_peft_model_state_dict  # noqa: F401
except Exception as _e:
    if INTERNET:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], capture_output=True)
        raise RuntimeError(f"PRE-FLIGHT: peft could not be imported ({_e!r}); it has now been UPDATED.\n"
                           "FIX: menu Run -> Restart & clear cell outputs, then Run All (only needed once).")
    raise RuntimeError(f"PRE-FLIGHT FAILED: peft cannot be imported ({_e!r}).\n"
                       "FIX: Settings -> Internet -> On, then Run All (the notebook will update peft).")
_ensure("nltk")          # optional: GLEU falls back to token-F1 if missing


def _fix_torchao():
    """Kaggle ships torchao 0.10, but recent peft refuses to build LoRA layers while an older torchao is installed
    ("ImportError: Found an incompatible version of torchao"). This notebook never uses torchao, so remove it.
    Works offline and needs no restart, because torchao itself is never imported."""
    try:
        from peft.import_utils import is_torchao_available
    except Exception:
        return
    try:
        is_torchao_available()
        return
    except ImportError as e:
        print(f"PRE-FLIGHT: {e}\n            -> uninstalling torchao (this notebook does not need it)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
    import importlib
    importlib.invalidate_caches()
    if hasattr(is_torchao_available, "cache_clear"):
        is_torchao_available.cache_clear()
    try:
        is_torchao_available()
        print("PRE-FLIGHT: torchao problem fixed")
    except ImportError:
        raise RuntimeError("PRE-FLIGHT FAILED: an incompatible torchao is installed and could not be removed.\n"
                           "FIX: add a first cell containing   !pip uninstall -y torchao   then menu Run -> "
                           "Restart & clear cell outputs, then Run All.")


_fix_torchao()

import numpy as np
import pandas as pd
import torch
import transformers
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict

try:
    from nltk.translate.gleu_score import sentence_gleu
except Exception:
    sentence_gleu = None

try:
    get_ipython  # noqa: F821
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ======================= THE SETTINGS YOU CHANGE ===========================
RUN_MODE   = "FULL"     # "SMOKE" (~10 min check)  ->  "QUICK" (~1.5 h, seed 0)  ->  "FULL" (~4 h, seeds 0-2)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"     # or "Qwen/Qwen2.5-1.5B-Instruct" (28 layers)
SHARED_INIT = True       # same LoRA A-initialisation for all languages of a seed (common practice; like your setup)
# ===========================================================================
RUN_MODE   = os.environ.get("KM_MODE", RUN_MODE)          # (environment overrides: only for automated testing)
MODEL_NAME = os.environ.get("KM_MODEL", MODEL_NAME)
OUT_DIR    = os.environ.get("KM_OUT", "/kaggle/working/kmerge_lab" if os.path.isdir("/kaggle/working") else "./kmerge_lab")
DTYPE_PREF = os.environ.get("KM_DTYPE", "auto")      # "auto" = bf16 on A100/H100/L4, else fp32 (safe on T4/P100)

LANGS = ["en", "de", "fr", "hi"]
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
TARGET_MODULES = ["q_proj", "v_proj"]
K_SLOTS = 2
DATA_SEED = 1234
N_BUFFER = 32            # unlabeled inputs per task for GRAM-U / RegMean activation Grams

MODE = {
    "SMOKE": dict(seeds=[0], n_train=48, n_val=12, n_test=20, epochs=2, lr=5e-4, batch=8,
                  pairs=[("en", "fr"), ("de", "fr")],
                  methods=["kmerge", "delta_avg", "ties", "tsv", "selfgram_cf", "gram_iso", "gramG", "regmean_cf", "gramU"],
                  seq_sets={"3lang": ["en", "de", "fr"]}, seq_orders_max=2,
                  seq_rules=["kmerge", "gramG"], seq_policies=["kmerge", "global_dist"],
                  share_methods=["split", "kshare"], share_rules=["kshare"], share_policies=["kmerge", "layer_loss"],
                  cap_sizes=[3], cap_merge=["kmerge", "gramU"], cap_share=["split", "kshare"]),
    "QUICK": dict(seeds=[0], n_train=128, n_val=40, n_test=100, epochs=3, lr=5e-4, batch=8,
                  pairs=None, methods=None,
                  seq_sets={"3lang": ["en", "de", "fr"], "4lang": ["en", "de", "fr", "hi"]}, seq_orders_max=None,
                  seq_rules=["kmerge", "delta_avg", "gramG", "gramU"],
                  seq_policies=["kmerge", "global_cos", "global_dist", "oracle"],
                  share_methods=["split", "kshare", "kshare_eq"], share_rules=["split", "kshare"],
                  share_policies=["kmerge", "global_loss", "layer_loss"],
                  cap_sizes=[3, 4], cap_merge=["kmerge", "delta_avg", "regmean_cf", "gramU", "gramG"],
                  cap_share=["split", "kshare", "kshare_eq"]),
    "FULL":  dict(seeds=[0, 1, 2], n_train=128, n_val=40, n_test=100, epochs=3, lr=5e-4, batch=8,
                  pairs=None, methods=None,
                  seq_sets={"3lang": ["en", "de", "fr"], "4lang": ["en", "de", "fr", "hi"]}, seq_orders_max=None,
                  seq_rules=["kmerge", "delta_avg", "ties", "tsv", "gramG", "gramU"],
                  seq_policies=["kmerge", "global_cos", "global_dist", "oracle"],
                  share_methods=["split", "kshare", "kshare_eq"], share_rules=["split", "kshare"],
                  share_policies=["kmerge", "global_loss", "layer_loss"],
                  cap_sizes=[3, 4], cap_merge=["kmerge", "delta_avg", "regmean_cf", "gramU", "gramG"],
                  cap_share=["split", "kshare", "kshare_eq"]),
}[RUN_MODE]
SEEDS = MODE["seeds"]
N_TRAIN, N_VAL, N_TEST = MODE["n_train"], MODE["n_val"], MODE["n_test"]
PAIRS = MODE["pairs"] or list(itertools.combinations(LANGS, 2))

# ---------------- PRE-FLIGHT CHECK: stops early with a plain-English fix ----------------
def _expected_hidden(name):
    m = re.search(r"(\d+(?:\.\d+)?)[bB]", name.split("/")[-1])
    return {"0.5": 896, "1.5": 1536, "3": 2048, "7": 3584}.get(m.group(1)) if m else None


def find_local_model(name, root="/kaggle/input"):
    """Find the model if it was attached with Add Input -> Models (works with Internet OFF)."""
    if os.path.isdir(name):
        return name
    if not os.path.isdir(root):
        return None
    want, inst = _expected_hidden(name), "instruct" in name.lower()
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - root.count(os.sep) >= 7:
            dirnames[:] = []
            continue
        if "config.json" in filenames and any(f.endswith((".safetensors", ".bin")) for f in filenames):
            try:
                cfg = json.load(open(os.path.join(dirpath, "config.json")))
            except Exception:
                continue
            if not str(cfg.get("model_type", "")).startswith("qwen2"):
                continue
            if want and cfg.get("hidden_size") != want:
                continue
            if inst and "instruct" not in dirpath.lower():
                continue
            return dirpath
    return None


_problems = []
if os.path.isdir("/kaggle") and not torch.cuda.is_available():
    _problems.append("No GPU is switched on.  FIX: right panel -> Settings -> Accelerator -> 'GPU T4 x2', then Run All.")
MODEL_PATH = find_local_model(MODEL_NAME)
if MODEL_PATH is None:
    if INTERNET:
        MODEL_PATH = MODEL_NAME
    else:
        _problems.append(
            "Internet is OFF and the model is not attached.  FIX one of:\n"
            "   (a) right panel -> Settings -> Internet -> On (needs a phone-verified Kaggle account), or\n"
            "   (b) right panel -> Add Input -> Models -> search 'Qwen2.5' -> Framework 'Transformers' -> "
            f"Variation '{MODEL_NAME.split('/')[-1].replace('Qwen2.5-', '').lower()}' -> Add. The notebook finds it automatically.")
_tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
if _tv < (4, 40):
    _problems.append(f"transformers {transformers.__version__} is too old.  FIX: Internet On, then add a cell "
                     "'!pip install -U transformers peft' at the top and Run All.")
print("PRE-FLIGHT | python", sys.version.split()[0], "| torch", torch.__version__, "| transformers",
      transformers.__version__, "| peft", peft.__version__, "| internet", "ON" if INTERNET else "OFF",
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| model from", MODEL_PATH)
if _problems:
    raise RuntimeError("PRE-FLIGHT FAILED:\n - " + "\n - ".join(_problems))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DTYPE_PREF == "auto":
    DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float32
else:
    DTYPE = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[DTYPE_PREF]
EVAL_BATCH = 48 if DEVICE.type == "cuda" else 8

MODEL_TAG = re.sub(r"[^A-Za-z0-9]+", "_", MODEL_NAME.strip("/").split("/")[-1])
# QUICK and FULL share data + training settings, so they share one cache: FULL reuses everything QUICK computed.
CFG_TAG = (f"{MODEL_TAG}_tr{N_TRAIN}-va{N_VAL}-te{N_TEST}_ep{MODE['epochs']}_lr{MODE['lr']}"
           f"_{'sharedinit' if SHARED_INIT else 'indepinit'}")
RUN_DIR = os.path.join(OUT_DIR, CFG_TAG)
ADAPTER_DIR = os.path.join(OUT_DIR, "adapters", CFG_TAG)
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(ADAPTER_DIR, exist_ok=True)

T0 = time.time()
def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f} min]", *a, flush=True)


def restore_from_inputs(root=os.environ.get("KM_INPUT", "/kaggle/input")):
    """RESUME: attach the output of an earlier run (right panel -> Add Input -> Your Work -> this notebook).
    Everything it already finished for the SAME settings (adapters, evaluations, results) is copied here and skipped,
    e.g. FULL reuses all of QUICK's seed-0 work. Files that already exist here are never overwritten."""
    import shutil
    if not os.path.isdir(root):
        return
    for dirpath, dirnames, _ in os.walk(root, followlinks=True):
        if dirpath.count(os.sep) - root.count(os.sep) >= 6:
            dirnames[:] = []
            continue
        if os.path.basename(dirpath) != "kmerge_lab" or os.path.abspath(dirpath) == os.path.abspath(OUT_DIR):
            continue
        dirnames[:] = []
        copied = 0
        for src, dst in ((os.path.join(dirpath, CFG_TAG), RUN_DIR), (os.path.join(dirpath, "adapters", CFG_TAG), ADAPTER_DIR)):
            if not os.path.isdir(src):
                continue
            for fn in sorted(os.listdir(src)):
                s, d = os.path.join(src, fn), os.path.join(dst, fn)
                if os.path.isfile(s) and not os.path.exists(d):
                    shutil.copy2(s, d)
                    os.chmod(d, 0o644)
                    copied += 1
        if copied:
            log(f"RESUME: copied {copied} cached files from {dirpath}; finished work will be skipped")


restore_from_inputs()

log(f"RUN_MODE={RUN_MODE} | model={MODEL_NAME} | device={DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'}) | dtype={DTYPE}")
log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | out={RUN_DIR}")

In [ ]:
# ============================================================================
# 2. DATA: templated grammar correction for EN / DE / FR / HI
# ============================================================================
EN_S3 = ["He", "She", "My brother", "My sister", "The teacher", "Our neighbour", "The doctor", "My friend", "The boy", "The girl"]
EN_PL = ["They", "We", "My parents", "The students", "Our neighbours", "The children", "My friends", "The doctors"]
EN_V = [("play", "plays", "played", ["football", "the piano", "chess"]),
        ("watch", "watches", "watched", ["the news", "a movie", "the birds"]),
        ("read", "reads", "read", ["a book", "the newspaper", "a story"]),
        ("cook", "cooks", "cooked", ["dinner", "rice", "pasta"]),
        ("write", "writes", "wrote", ["a letter", "an email", "a poem"]),
        ("drink", "drinks", "drank", ["coffee", "tea", "water"]),
        ("visit", "visits", "visited", ["the museum", "the market", "the library"]),
        ("clean", "cleans", "cleaned", ["the kitchen", "the room", "the car"]),
        ("study", "studies", "studied", ["English", "maths", "history"]),
        ("wash", "washes", "washed", ["the dishes", "the car", "the clothes"]),
        ("eat", "eats", "ate", ["an apple", "breakfast", "lunch"]),
        ("buy", "buys", "bought", ["bread", "new shoes", "vegetables"])]
EN_T_PRES = ["every day", "on Sundays", "every morning", "in the evening"]
EN_T_PAST = ["Yesterday", "Last week", "Last night"]

PERS = ["1s", "2s", "3s", "1p", "2p", "3p"]
DE_SUBJ = [("Ich", "1s"), ("Du", "2s"), ("Er", "3s"), ("Maria", "3s"), ("Der Lehrer", "3s"), ("Mein Bruder", "3s"),
           ("Wir", "1p"), ("Ihr", "2p"), ("Die Kinder", "3p"), ("Meine Eltern", "3p")]
DE_V = [("spielen", ["spiele", "spielst", "spielt", "spielen", "spielt", "spielen"], ["Fußball", "Gitarre", "Schach"]),
        ("lernen", ["lerne", "lernst", "lernt", "lernen", "lernt", "lernen"], ["Deutsch", "Mathematik", "Englisch"]),
        ("kaufen", ["kaufe", "kaufst", "kauft", "kaufen", "kauft", "kaufen"], ["Brot", "Milch", "Obst"]),
        ("kochen", ["koche", "kochst", "kocht", "kochen", "kocht", "kochen"], ["Suppe", "Nudeln", "Reis"]),
        ("trinken", ["trinke", "trinkst", "trinkt", "trinken", "trinkt", "trinken"], ["Kaffee", "Tee", "Wasser"]),
        ("hören", ["höre", "hörst", "hört", "hören", "hört", "hören"], ["Musik", "Radio"]),
        ("machen", ["mache", "machst", "macht", "machen", "macht", "machen"], ["Hausaufgaben", "Sport"]),
        ("besuchen", ["besuche", "besuchst", "besucht", "besuchen", "besucht", "besuchen"], ["das Museum", "die Großeltern"]),
        ("haben", ["habe", "hast", "hat", "haben", "habt", "haben"], ["einen Hund", "eine Katze", "ein Auto", "keine Zeit"]),
        ("sein", ["bin", "bist", "ist", "sind", "seid", "sind"], ["müde", "glücklich", "zu Hause"]),
        ("lesen", ["lese", "liest", "liest", "lesen", "lest", "lesen"], ["ein Buch", "die Zeitung"]),
        ("essen", ["esse", "isst", "isst", "essen", "esst", "essen"], ["einen Apfel", "Brot", "Pizza"]),
        ("fahren", ["fahre", "fährst", "fährt", "fahren", "fahrt", "fahren"], ["nach Berlin", "mit dem Bus"]),
        ("sprechen", ["spreche", "sprichst", "spricht", "sprechen", "sprecht", "sprechen"], ["Deutsch", "Englisch", "Französisch"])]

FR_SUBJ = [("Tu", "2s"), ("Il", "3s"), ("Elle", "3s"), ("Marie", "3s"), ("Mon frère", "3s"), ("Nous", "1p"),
           ("Vous", "2p"), ("Ils", "3p"), ("Elles", "3p"), ("Les enfants", "3p"), ("Mes amis", "3p")]
FR_V = [("parler", ["parle", "parles", "parle", "parlons", "parlez", "parlent"], ["français", "anglais", "espagnol"]),
        ("regarder", ["regarde", "regardes", "regarde", "regardons", "regardez", "regardent"], ["la télévision", "un film"]),
        ("jouer", ["joue", "joues", "joue", "jouons", "jouez", "jouent"], ["au football", "du piano", "aux cartes"]),
        ("écouter", ["écoute", "écoutes", "écoute", "écoutons", "écoutez", "écoutent"], ["la radio", "de la musique"]),
        ("aimer", ["aime", "aimes", "aime", "aimons", "aimez", "aiment"], ["le chocolat", "la musique", "le cinéma"]),
        ("travailler", ["travaille", "travailles", "travaille", "travaillons", "travaillez", "travaillent"], ["à Paris", "le samedi"]),
        ("chanter", ["chante", "chantes", "chante", "chantons", "chantez", "chantent"], ["une chanson"]),
        ("habiter", ["habite", "habites", "habite", "habitons", "habitez", "habitent"], ["à Lyon", "en France"]),
        ("porter", ["porte", "portes", "porte", "portons", "portez", "portent"], ["une veste", "un chapeau"]),
        ("préparer", ["prépare", "prépares", "prépare", "préparons", "préparez", "préparent"], ["le dîner", "un gâteau"]),
        ("acheter", ["achète", "achètes", "achète", "achetons", "achetez", "achètent"], ["du pain", "des fruits", "un livre"]),
        ("manger", ["mange", "manges", "mange", "mangeons", "mangez", "mangent"], ["une pomme", "du pain"]),
        ("avoir", ["ai", "as", "a", "avons", "avez", "ont"], ["un chien", "une voiture", "faim"]),
        ("être", ["suis", "es", "est", "sommes", "êtes", "sont"], ["à la maison", "en retard"]),
        ("aller", ["vais", "vas", "va", "allons", "allez", "vont"], ["au marché", "à l'école", "au cinéma"]),
        ("faire", ["fais", "fais", "fait", "faisons", "faites", "font"], ["du sport", "les courses"]),
        ("prendre", ["prends", "prends", "prend", "prenons", "prenez", "prennent"], ["le bus", "un café", "le train"])]

HI_SUBJ = [("राम", "m", "s"), ("सीता", "f", "s"), ("लड़का", "m", "s"), ("लड़की", "f", "s"), ("मेरा भाई", "m", "s"),
           ("मेरी बहन", "f", "s"), ("मेरी माँ", "f", "s"), ("वे", "m", "p"), ("हम", "m", "p"), ("लड़के", "m", "p"),
           ("लड़कियाँ", "f", "p"), ("बच्चे", "m", "p")]
HI_V = [("खा", ["खाना", "सेब", "रोटी"]), ("पी", ["पानी", "चाय", "दूध"]), ("पढ़", ["किताब", "अखबार", "कहानी"]),
        ("लिख", ["पत्र", "कहानी", "कविता"]), ("खेल", ["क्रिकेट", "फुटबॉल", "शतरंज"]), ("गा", ["गाना", "गीत"]),
        ("बोल", ["हिंदी", "अंग्रेजी"]), ("बना", ["खाना", "चाय"]), ("सुन", ["संगीत", "रेडियो", "गाना"]),
        ("सीख", ["हिंदी", "गणित"]), ("जा", ["स्कूल", "बाजार", "दफ्तर"])]
HI_SUF = {("m", "s"): "ता", ("f", "s"): "ती", ("m", "p"): "ते", ("f", "p"): "ती"}


def _lc_first(s):
    return s[0].lower() + s[1:]


def gen_en(rng):
    pool = []
    for subj in EN_S3:
        for base, v3, past, objs in EN_V:
            for o in objs:
                for t in EN_T_PRES:
                    pool.append((f"{subj} {v3} {o} {t}.", f"{subj} {base} {o} {t}.", "agreement"))
    for subj in EN_PL:
        for base, v3, past, objs in EN_V:
            for o in objs:
                for t in EN_T_PRES:
                    pool.append((f"{subj} {base} {o} {t}.", f"{subj} {v3} {o} {t}.", "agreement"))
    for subj, is3 in [(s, True) for s in EN_S3] + [(s, False) for s in EN_PL]:
        for base, v3, past, objs in EN_V:
            wrong = v3 if is3 else base
            if wrong == past:
                continue
            for o in objs:
                for T in EN_T_PAST:
                    pool.append((f"{T} {_lc_first(subj)} {past} {o}.", f"{T} {_lc_first(subj)} {wrong} {o}.", "tense"))
    return pool


def _conj_pool(subjects, verbs, rng, p_inf):
    pool = []
    for subj, p in subjects:
        pi = PERS.index(p)
        for inf, forms, objs in verbs:
            right = forms[pi]
            others = sorted({f for f in forms if f != right})
            for o in objs:
                use_inf = (rng.random() < p_inf) and inf != right
                wrong = inf if use_inf else rng.choice(others)
                pool.append((f"{subj} {right} {o}.", f"{subj} {wrong} {o}.", "infinitive" if use_inf else "person"))
    return pool


def gen_de(rng):
    return _conj_pool(DE_SUBJ, DE_V, rng, p_inf=0.3)


def gen_fr(rng):
    return _conj_pool(FR_SUBJ, FR_V, rng, p_inf=0.5)


def gen_hi(rng):
    pool = []
    for subj, g, n in HI_SUBJ:
        aux = "है" if n == "s" else "हैं"
        suf = HI_SUF[(g, n)]
        for stem, objs in HI_V:
            for o in objs:
                tgt = f"{subj} {o} {stem}{suf} {aux}।"
                kind = rng.choice(["gender", "number", "aux"])
                if kind == "gender":
                    wsuf = {"ता": "ती", "ती": "ता", "ते": "ती"}[suf]
                    err = f"{subj} {o} {stem}{wsuf} {aux}।"
                elif kind == "number":
                    wn = "p" if n == "s" else "s"
                    err = f"{subj} {o} {stem}{HI_SUF[(g, wn)]} {'है' if wn == 's' else 'हैं'}।"
                else:
                    err = f"{subj} {o} {stem}{suf} {'हैं' if aux == 'है' else 'है'}।"
                if err != tgt:
                    pool.append((tgt, err, kind))
    return pool


GENERATORS = {"en": gen_en, "de": gen_de, "fr": gen_fr, "hi": gen_hi}


def make_splits(lang):
    rng = random.Random(DATA_SEED + LANGS.index(lang))
    pool = GENERATORS[lang](rng)
    seen, uniq = set(), []
    for tgt, err, kind in pool:          # one example per target sentence -> no target is shared across splits
        if tgt not in seen:
            seen.add(tgt); uniq.append((tgt, err, kind))
    rng.shuffle(uniq)
    need = N_TRAIN + N_VAL + N_TEST
    assert len(uniq) >= need, f"{lang}: pool {len(uniq)} < needed {need}"
    sizes = {"train": N_TRAIN, "val": N_VAL, "test": N_TEST}
    out, start = {}, 0
    for split, size in sizes.items():
        rows = []
        for i, (tgt, err, kind) in enumerate(uniq[start:start + size]):
            if i % 5 == 0:   # 20% already-correct sentences: the model must leave them unchanged
                rows.append({"src": tgt, "tgt": tgt, "kind": "correct"})
            else:
                rows.append({"src": err, "tgt": tgt, "kind": kind})
        rng.shuffle(rows)
        out[split] = rows
        start += size
    return out, len(uniq)


DATA, POOL_SIZES = {}, {}
for _l in LANGS:
    DATA[_l], POOL_SIZES[_l] = make_splits(_l)
for _l in LANGS:
    tr = {r["tgt"] for r in DATA[_l]["train"]}
    va = {r["tgt"] for r in DATA[_l]["val"]}
    te = {r["tgt"] for r in DATA[_l]["test"]}
    assert not (tr & va) and not (tr & te) and not (va & te), f"overlap in {_l}"
    ex = [r for r in DATA[_l]["test"] if r["kind"] != "correct"][:2]
    log(f"{_l}: pool={POOL_SIZES[_l]} targets | train/val/test={N_TRAIN}/{N_VAL}/{N_TEST} | e.g. "
        + " || ".join(f"{e['src']}  ->  {e['tgt']}" for e in ex))

In [ ]:
# ============================================================================
# 3. MODEL, LORA TRAINING, EVALUATION
# ============================================================================
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
END_TOKEN = "<|im_end|>" if "<|im_end|>" in tok.get_vocab() else tok.eos_token
EOS_IDS = sorted({i for i in [tok.convert_tokens_to_ids(END_TOKEN), tok.eos_token_id] if i is not None and i >= 0})

try:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=DTYPE)
except TypeError:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=DTYPE)
base_model.to(DEVICE)
model = get_peft_model(base_model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                              target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM"))
model.eval()

MOD_MAP = OrderedDict((n, m) for n, m in model.named_modules()
                      if hasattr(m, "lora_A") and hasattr(m.lora_A, "keys") and "default" in m.lora_A)
N_LAYERS = 1 + max(int(re.search(r"layers\.(\d+)\.", k).group(1)) for k in MOD_MAP)
log(f"LoRA modules: {len(MOD_MAP)} ({N_LAYERS} layers x {TARGET_MODULES})")

SYSTEM = "You are a careful grammar-correction assistant."
INSTR = ("Correct the grammar of this sentence. If it is already correct, repeat it unchanged. "
         "Reply with the corrected sentence only.\n\n")


def build_prompt(src):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": INSTR + src}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


PROMPTS = {l: {s: [build_prompt(r["src"]) for r in DATA[l][s]] for s in ("train", "val", "test")} for l in LANGS}
MAX_NEW = {l: max(len(tok(r["tgt"] + END_TOKEN, add_special_tokens=False).input_ids) for s in ("val", "test")
                  for r in DATA[l][s]) + 8 for l in LANGS}
GEN_CFGS = {l: GenerationConfig(max_new_tokens=MAX_NEW[l], do_sample=False, num_beams=1, repetition_penalty=1.0,
                                pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS) for l in LANGS}


def init_lora(seed):
    """PEFT's default LoRA init (kaiming-uniform A with a=sqrt(5), zero B), from an explicit generator."""
    g = torch.Generator().manual_seed(int(seed))
    with torch.no_grad():
        for mk, m in MOD_MAP.items():
            A, B = m.lora_A["default"].weight, m.lora_B["default"].weight
            bound = 1.0 / math.sqrt(A.shape[1])
            A.copy_(((torch.rand(A.shape, generator=g) * 2 - 1) * bound).to(A.dtype))
            B.zero_()


def fac_to_model(fac):
    with torch.no_grad():
        for mk, (B, A) in fac.items():
            m = MOD_MAP[mk]
            m.lora_A["default"].weight.copy_(A.to(m.lora_A["default"].weight.dtype))
            m.lora_B["default"].weight.copy_(B.to(m.lora_B["default"].weight.dtype))


def state_to_fac(state):
    fac = OrderedDict()
    for mk in MOD_MAP:
        fac[mk] = (state[mk + ".lora_B.weight"].float().to(DEVICE), state[mk + ".lora_A.weight"].float().to(DEVICE))
    return fac


def _tokenize_train(lang):
    rows = []
    for p, r in zip(PROMPTS[lang]["train"], DATA[lang]["train"]):
        pi = tok(p, add_special_tokens=False).input_ids
        ti = tok(r["tgt"] + END_TOKEN, add_special_tokens=False).input_ids
        rows.append((pi + ti, [-100] * len(pi) + ti))
    return rows


def train_adapter(lang, seed):
    path = os.path.join(ADAPTER_DIR, f"seed{seed}_{lang}.pt")
    if os.path.exists(path):
        return torch.load(path, map_location="cpu")
    init_lora(seed if SHARED_INIT else seed * 1000 + LANGS.index(lang) + 1)
    data = _tokenize_train(lang)
    rng = random.Random(seed * 7919 + LANGS.index(lang))
    params = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.AdamW(params, lr=MODE["lr"], weight_decay=0.0)
    steps = MODE["epochs"] * math.ceil(len(data) / MODE["batch"])
    warm = max(1, steps // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, (steps - s) / max(1, steps - warm)))
    model.train()
    t0, losses = time.time(), []
    for ep in range(MODE["epochs"]):
        order = list(range(len(data))); rng.shuffle(order)
        for i in range(0, len(order), MODE["batch"]):
            batch = [data[j] for j in order[i:i + MODE["batch"]]]
            L = max(len(x[0]) for x in batch)
            ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
            lab = torch.full((len(batch), L), -100, dtype=torch.long)
            att = torch.zeros((len(batch), L), dtype=torch.long)
            for k, (x, y) in enumerate(batch):
                ids[k, :len(x)] = torch.tensor(x); lab[k, :len(y)] = torch.tensor(y); att[k, :len(x)] = 1
            out = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE), labels=lab.to(DEVICE))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            losses.append(out.loss.item())
    model.eval()
    state = {k: v.detach().float().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}
    torch.save(state, path)
    log(f"trained {lang} seed {seed}: {steps} steps, loss {np.mean(losses[:3]):.3f} -> {np.mean(losses[-3:]):.3f} "
        f"({time.time() - t0:.0f}s)")
    return state


# ---------------- metrics ----------------
TOK_RE = re.compile(r"[^\s.,!?;:।]+|[.,!?;:।]")
DEVA_RE = re.compile(r"[ऀ-ॿ]")
LATIN_RE = re.compile(r"[A-Za-z]")
LATIN_WORD_RE = re.compile(r"[a-zäöüßàâçéèêëîïôûùœ']+")
FUNC_WORDS = {"en": {"the", "he", "she", "is", "are", "they", "we", "my", "his", "her", "and", "to", "goes", "plays"},
              "de": {"der", "die", "das", "und", "ist", "er", "sie", "hat", "ein", "eine", "nicht", "ich", "wir", "mein", "meine"},
              "fr": {"le", "la", "les", "et", "est", "il", "elle", "un", "une", "des", "du", "nous", "vous", "mon", "mes"}}


def _norm(s):
    return " ".join(str(s).strip().split())


def _toks(s):
    return TOK_RE.findall(s)


def tok_f1(p, t):
    pc, tc = Counter(_toks(p)), Counter(_toks(t))
    ov = sum((pc & tc).values())
    if ov == 0:
        return 0.0
    pr, rc = ov / max(1, sum(pc.values())), ov / max(1, sum(tc.values()))
    return 2 * pr * rc / (pr + rc)


def gleu(p, t):
    pt, tt = _toks(p), _toks(t)
    if not pt or not tt:
        return 0.0
    if sentence_gleu is not None:
        return float(sentence_gleu([tt], pt, min_len=1, max_len=4))
    return tok_f1(p, t)


def is_leak(pred, src, lang):
    if lang == "hi":
        return bool(LATIN_RE.search(pred))
    if DEVA_RE.search(pred):
        return True
    new_words = set(LATIN_WORD_RE.findall(pred.lower())) - set(LATIN_WORD_RE.findall(src.lower()))
    foreign = set().union(*[FUNC_WORDS[l] for l in ("en", "de", "fr") if l != lang])
    return bool(new_words & foreign)


def score(preds, rows, lang):
    agg = defaultdict(float); n_err = n_cor = 0
    for p, r in zip(preds, rows):
        P, T, S = _norm(p), _norm(r["tgt"]), _norm(r["src"])
        agg["em"] += float(P == T); agg["f1"] += tok_f1(P, T); agg["gleu"] += gleu(P, T)
        agg["edit"] += difflib.SequenceMatcher(None, P, T).ratio()
        agg["leak"] += float(is_leak(P, S, lang)); agg["empty"] += float(P == "")
        if r["kind"] == "correct":
            n_cor += 1; agg["over"] += float(P != S)
        else:
            n_err += 1; agg["copy"] += float(P == S)
    n = max(1, len(rows))
    out = {k: agg[k] / n for k in ("em", "f1", "gleu", "edit", "leak", "empty")}
    out["copy"] = agg["copy"] / max(1, n_err)
    out["over"] = agg["over"] / max(1, n_cor)
    return out


# ---------------- generation + persistent eval cache ----------------
CACHE_PATH = os.path.join(RUN_DIR, "eval_cache.jsonl")
EVAL_CACHE = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        for line in f:
            try:
                d = json.loads(line); EVAL_CACHE[d["k"]] = d["m"]
            except Exception:
                pass
PRED_PATH = os.path.join(RUN_DIR, "predictions_test.csv")


@torch.no_grad()
def generate(prompts, lang):
    outs = []
    for i in range(0, len(prompts), EVAL_BATCH):
        enc = tok(prompts[i:i + EVAL_BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        gen = model.generate(**enc, generation_config=GEN_CFGS[lang])
        texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        outs += [t.strip().split("\n")[0].strip() for t in texts]
    return outs


def evaluate(key, fac, lang, split):
    """fac: adapter factors, a zero-arg function returning them (lazy), or None for the base model."""
    ck = f"{key}|{lang}|{split}"
    if ck in EVAL_CACHE:
        return EVAL_CACHE[ck]
    if fac is None:
        with model.disable_adapter():
            preds = generate(PROMPTS[lang][split], lang)
    else:
        fac_to_model(fac() if callable(fac) else fac)
        preds = generate(PROMPTS[lang][split], lang)
    m = score(preds, DATA[lang][split], lang)
    EVAL_CACHE[ck] = m
    with open(CACHE_PATH, "a") as f:
        f.write(json.dumps({"k": ck, "m": m}) + "\n")
    if split == "test":
        pd.DataFrame([{"key": key, "lang": lang, "kind": r["kind"], "src": r["src"], "tgt": r["tgt"], "pred": p}
                      for p, r in zip(preds, DATA[lang][split])]).to_csv(
            PRED_PATH, mode="a", header=not os.path.exists(PRED_PATH), index=False)
    return m


# ---------------- activation Grams (for GRAM-U / GRAM-G and RegMean) ----------------
LANG_NAMES = {"en": "English", "de": "German", "fr": "French", "hi": "Hindi"}


@torch.no_grad()
def generate_pseudo_inputs(lang, n=N_BUFFER):
    """GRAM-G: no user data at all. The BASE model writes n sentences from the task description only
    (K-Merge already assumes each adapter's task is known, because it routes inputs by task)."""
    msgs = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"Write one short, simple everyday sentence in {LANG_NAMES[lang]}. "
                                        f"Reply with the sentence only."}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    cfg = GenerationConfig(max_new_tokens=48, do_sample=True, temperature=1.0, top_p=0.95, repetition_penalty=1.0,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)
    torch.manual_seed(DATA_SEED + LANGS.index(lang))
    outs = []
    with model.disable_adapter():
        for i in range(0, n, EVAL_BATCH):
            enc = tok([prompt] * min(EVAL_BATCH, n - i), return_tensors="pt", padding=True,
                      add_special_tokens=False).to(DEVICE)
            gen = model.generate(**enc, generation_config=cfg)
            for t in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True):
                t = t.strip().split("\n")[0].strip().strip('"“”*').strip()
                outs.append(t if t else "...")
    return outs


@torch.no_grad()
def collect_grams(sources):
    """Average x x^T of each LoRA module's input over GEC prompts built from `sources` (base model, no labels)."""
    grams = {mk: None for mk in MOD_MAP}
    counts = {mk: 0 for mk in MOD_MAP}
    state = {"mask": None}
    hooks = []
    for mk, m in MOD_MAP.items():
        def hook(mod, args, mk=mk):
            x = args[0]
            xv = x[state["mask"]].float() if state["mask"] is not None else x.reshape(-1, x.shape[-1]).float()
            g = xv.T @ xv
            grams[mk] = g if grams[mk] is None else grams[mk] + g
            counts[mk] += xv.shape[0]
        hooks.append(m.register_forward_pre_hook(hook))
    prompts = [build_prompt(s) for s in sources]
    try:
        with model.disable_adapter():
            for i in range(0, len(prompts), EVAL_BATCH):
                enc = tok(prompts[i:i + EVAL_BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
                state["mask"] = enc["attention_mask"].bool()
                model(**enc)
    finally:
        for h in hooks:
            h.remove()
    return {mk: grams[mk] / max(1, counts[mk]) for mk in MOD_MAP}


def prep_gram(C, beta=0.9):
    """RegMean-style shrinkage of off-diagonals, then normalise to mean eigenvalue 1."""
    C = (C + C.T) / 2
    C = beta * C + (1 - beta) * torch.diag(torch.diag(C))
    return C / (torch.trace(C) / C.shape[0]).clamp_min(1e-12)

In [ ]:
# ============================================================================
# 4. MERGE LIBRARY: every method merges two adapters into ONE rank-r adapter
#    adapter = {module_key: (B [d_out x r], A [r x d_in])}, delta W = B @ A
# ============================================================================
R = LORA_R


def lr_svd(L, Rt):
    """SVD of M = L @ Rt without forming M (exact). Returns U, S, V with M = U diag(S) V^T."""
    Ql, Rl = torch.linalg.qr(L)
    Qr, Rr = torch.linalg.qr(Rt.T)
    U, S, Vh = torch.linalg.svd(Rl @ Rr.T, full_matrices=False)
    return Ql @ U, S, Qr @ Vh.T


def _to_rank(U, S, V, r=R):
    k = min(r, S.numel())
    s = S[:k].clamp_min(0).sqrt()
    B, A = U[:, :k] * s, s[:, None] * V[:, :k].T
    if k < r:
        B = torch.cat([B, B.new_zeros(B.shape[0], r - k)], 1)
        A = torch.cat([A, A.new_zeros(r - k, A.shape[1])], 0)
    return B, A


def trunc_lr(L, Rt, r=R):
    U, S, V = lr_svd(L, Rt)
    tot, kept = float((S ** 2).sum()), float((S[:r] ** 2).sum())
    B, A = _to_rank(U, S, V, r)
    return B, A, tot - kept, tot


def trunc_dense(M, r=R):
    tot = float((M ** 2).sum())
    q = min(r + 10, min(M.shape))
    torch.manual_seed(0)
    U, S, V = torch.svd_lowrank(M, q=q, niter=4)
    B, A = _to_rank(U, S, V, r)
    return B, A, tot - float((S[:r] ** 2).sum()), tot


def ip(a, b):
    """<B_a A_a, B_b A_b>_F computed in r x r."""
    return ((a[0].T @ b[0]) * (a[1] @ b[1].T)).sum()


def _dense(a):
    return a[0] @ a[1]


def _cat(a, b, ca, cb):
    """Low-rank factors of ca * dW_a + cb * dW_b."""
    return torch.cat([ca * a[0], cb * b[0]], 1), torch.cat([a[1], b[1]], 0)


# ---------- family A: averaging / scaling ----------
def mm_kmerge(a, b, wa, wb, hp, ctx):
    """K-Merge Eq. 8: history-weighted running average applied to the LoRA matrices A and B."""
    W = wa + wb
    return (wa * a[0] + wb * b[0]) / W, (wa * a[1] + wb * b[1]) / W, 0.0, 0.0


def mm_delta_avg(a, b, wa, wb, hp, ctx):
    W = wa + wb
    return trunc_lr(*_cat(a, b, wa / W, wb / W))


def mm_task_arith(a, b, wa, wb, hp, ctx):
    c = 2 * hp["lam"] / (wa + wb)
    return trunc_lr(*_cat(a, b, c * wa, c * wb))


def mm_slerp(a, b, wa, wb, hp, ctx):
    na, nb = ip(a, a).sqrt(), ip(b, b).sqrt()
    cos = float((ip(a, b) / (na * nb).clamp_min(1e-30)).clamp(-1, 1))
    th, t = math.acos(cos), wb / (wa + wb)
    if th < 1e-3:
        ca, cb = 1 - t, t
    else:
        ca, cb = math.sin((1 - t) * th) / math.sin(th), math.sin(t * th) / math.sin(th)
    return trunc_lr(*_cat(a, b, ca, cb))


# ---------- family B: sparsify / sign ----------
def _trim(T, keep):
    k = max(1, int(round(keep * T.numel())))
    if k >= T.numel():
        return T
    thr = T.abs().flatten().kthvalue(T.numel() - k + 1).values
    return T * (T.abs() >= thr)


def _ties_core(Ta, Tb, wa, wb, keep):
    Ta, Tb = _trim(Ta, keep), _trim(Tb, keep)
    el = torch.sign(wa * Ta + wb * Tb)
    ma = ((torch.sign(Ta) == el) & (Ta != 0)).float()
    mb = ((torch.sign(Tb) == el) & (Tb != 0)).float()
    den = wa * ma + wb * mb
    return torch.where(den > 0, (wa * Ta * ma + wb * Tb * mb) / den.clamp_min(1e-12), torch.zeros_like(Ta))


def mm_ties(a, b, wa, wb, hp, ctx):
    return trunc_dense(_ties_core(_dense(a), _dense(b), wa, wb, hp["keep"]))


def mm_dare(a, b, wa, wb, hp, ctx):
    p = 1 - hp["drop"]
    g = torch.Generator(device=a[0].device).manual_seed(int(ctx["seed"]))
    Ta, Tb = _dense(a), _dense(b)
    ma = (torch.rand(Ta.shape, generator=g, device=Ta.device) < p).float() / p
    mb = (torch.rand(Tb.shape, generator=g, device=Tb.device) < p).float() / p
    return trunc_dense((wa * Ta * ma + wb * Tb * mb) / (wa + wb))


def mm_magmax(a, b, wa, wb, hp, ctx):
    Ta, Tb = _dense(a), _dense(b)
    return trunc_dense(torch.where(Ta.abs() >= Tb.abs(), Ta, Tb))


# ---------- family C: subspace / SVD ----------
def mm_knots_ties(a, b, wa, wb, hp, ctx):
    """KnOTS: joint SVD of [dW_a, dW_b] = U S [V_a^T, V_b^T]; TIES-merge the task-specific V's; rebuild with U S."""
    r, n = a[0].shape[1], a[1].shape[1]
    Q, Rq = torch.linalg.qr(torch.cat([a[0], b[0]], 1))
    Z = torch.cat([Rq[:, :r] @ a[1], Rq[:, r:] @ b[1]], 1)
    Uz, S, Vh = torch.linalg.svd(Z, full_matrices=False)
    Vm = _ties_core(Vh[:, :n], Vh[:, n:], wa, wb, hp["keep"])
    return trunc_lr(Q @ (Uz * S), Vm)


def _polar(M):
    P, _, Qh = torch.linalg.svd(M, full_matrices=False)
    return P @ Qh


def mm_tsv(a, b, wa, wb, hp, ctx):
    """TSV-M: keep each task's top singular directions (rank budget split by history weight), decorrelate, sum."""
    Ua, Sa, Va = lr_svd(*a)
    Ub, Sb, Vb = lr_svd(*b)
    ka = min(max(int(round(R * wa / (wa + wb))), 1), R - 1); kb = R - ka
    U = _polar(torch.cat([Ua[:, :ka], Ub[:, :kb]], 1))
    V = _polar(torch.cat([Va[:, :ka], Vb[:, :kb]], 1))
    s = (hp.get("alpha", 1.0) * torch.cat([Sa[:ka], Sb[:kb]])).clamp_min(0).sqrt()
    return U * s, s[:, None] * V.T, 0.0, 0.0


def mm_iso_c(a, b, wa, wb, hp, ctx):
    """Iso-C: sum the task matrices, keep the top-r singular directions with a flat (isotropic) spectrum."""
    c = 2 / (wa + wb)
    U, S, V = lr_svd(*_cat(a, b, c * wa, c * wb))
    nz = S > 1e-6 * S.max().clamp_min(1e-30)
    sbar = float(S[nz].mean()) if nz.any() else 0.0
    k = min(R, int(nz.sum().item())) if nz.any() else 0
    s = torch.full((k,), math.sqrt(hp["alpha"] * sbar), device=U.device)
    return _to_rank(U[:, :k], s ** 2, V[:, :k]) + (0.0, 0.0)


# ---------- family D: conflict-aware (lab v1, from the earlier analysis) ----------
def overlap(a, b, k=4):
    Ua, Ub = lr_svd(*a)[0][:, :k], lr_svd(*b)[0][:, :k]
    return float(torch.linalg.svdvals(Ua.T @ Ub).mean())


def _ocp_LR(a, b, wa, wb, k=4):
    Ua = lr_svd(*a)[0][:, :k]
    Bb_orth = b[0] - Ua @ (Ua.T @ b[0])
    return torch.cat([a[0], (wb / (wa + wb)) * Bb_orth], 1), torch.cat([a[1], b[1]], 0)


def mm_layer_sel(a, b, wa, wb, hp, ctx):
    if overlap(a, b) < hp["th"]:
        return a[0].clone(), a[1].clone(), 0.0, 0.0
    return mm_delta_avg(a, b, wa, wb, hp, ctx)


def mm_ocp(a, b, wa, wb, hp, ctx):
    if overlap(a, b) < hp["th"]:
        return trunc_lr(*_ocp_LR(a, b, wa, wb))
    return mm_delta_avg(a, b, wa, wb, hp, ctx)


def mm_casc_soft(a, b, wa, wb, hp, ctx):
    s, W = overlap(a, b), wa + wb
    L1, R1 = _cat(a, b, wa / W, wb / W)
    L2, R2 = _ocp_LR(a, b, wa, wb)
    return trunc_lr(torch.cat([s * L1, (1 - s) * L2], 1), torch.cat([R1, R2], 0))


# ---------- family E/F: functional (Gram) merges, incl. GRAM ----------
def _eig_pos(G, tol=1e-6):
    lam, E = torch.linalg.eigh((G + G.T) / 2)
    keep = lam > tol * lam.max().clamp_min(1e-30)
    return E[:, keep], lam[keep]


def mm_gram_self(a, b, wa, wb, hp, ctx):
    """Data-free functional merge, solved exactly in the joint row space of the two adapters (dimension <= 2r).
    Input model of task t:  C_t = (1 - rho) * G_t / tr(G_t) + (rho / d_in) * I,   G_t = dW_t^T dW_t (self-Gram).
      rho = 0          -> pure self-Gram = WUDI's objective solved in closed form (selfgram_cf).
                          Unstable for LoRA with shared A-initialisation: the two row spaces almost coincide and the
                          exact solution exploits their tiny differences with huge coefficients.
      rho > 0          -> an isotropic input prior (same total energy as the footprint) regularises those directions.
    whiten=True  -> exact rank-r optimum of the objective (Eckart-Young in the C-metric); False -> plain SVD truncation.
    fair=True    -> task weights n_t / sqrt(E_t), E_t = ||dW_t C_t^{1/2}||^2. For non-overlapping tasks this makes the
                    relative damage of both tasks exactly equal (1/E_t over-corrects towards the smaller adapter)."""
    (Ba, Aa), (Bb, Ab) = a, b
    rho, d_in = float(hp.get("rho", 0.5)), Aa.shape[1]
    Q, _ = torch.linalg.qr(torch.cat([Aa, Ab], 0).T)          # d_in x 2r basis of both row spaces
    Ma, Mb = Ba @ (Aa @ Q), Bb @ (Ab @ Q)                      # dW_t in core coordinates
    k_ = Q.shape[1]
    eye = torch.eye(k_, device=Q.device, dtype=Q.dtype)
    Cs = []
    for M_ in (Ma, Mb):
        g_ = M_.T @ M_
        Cs.append((1 - rho) * g_ / torch.trace(g_).clamp_min(1e-30) + (rho / d_in) * eye)
    Ca, Cb = Cs
    if hp.get("fair", True):
        ca = wa / torch.trace(Ma @ Ca @ Ma.T).clamp_min(1e-30).sqrt()
        cb = wb / torch.trace(Mb @ Cb @ Mb.T).clamp_min(1e-30).sqrt()
    else:
        ca, cb = wa, wb
    E, lam = _eig_pos(ca * Ca + cb * Cb, tol=1e-9)
    NE = (ca * (Ma @ Ca) + cb * (Mb @ Cb)) @ E                 # X* G E = N E
    if not hp.get("whiten", True):
        return trunc_lr(NE / lam, E.T @ Q.T)                    # X* = N E diag(1/lam) E^T Q^T
    U, S, Vh = torch.linalg.svd(NE / lam.sqrt(), full_matrices=False)   # Z* = X* G^{1/2}
    tot, kept = float((S ** 2).sum()), float((S[:R] ** 2).sum())
    k = min(R, S.numel()); s = S[:k].sqrt()
    B = U[:, :k] * s
    A = ((s[:, None] * Vh[:k]) / lam.sqrt()[None, :]) @ (E.T @ Q.T)   # X = Z_r G^{-1/2} Q^T
    if k < R:
        B = torch.cat([B, B.new_zeros(B.shape[0], R - k)], 1); A = torch.cat([A, A.new_zeros(R - k, A.shape[1])], 0)
    return B, A, tot - kept, tot


def mm_gram_act(a, b, wa, wb, hp, ctx):
    """Functional merge with activation Grams C_t (RegMean objective), solved exactly.
    C_t comes from 32 real UNLABELED inputs (GRAM-U) or from 32 sentences the base model wrote from the task
    description alone (GRAM-G, fully data-free).
    whiten=False -> RegMean closed form + plain SVD truncation (regmean_cf / regmeanG_cf)
    whiten=True  -> exact rank-r optimum; fair as in mm_gram_self."""
    (Ba, Aa), (Bb, Ab) = a, b
    Ca, Cb = ctx["Ca"], ctx["Cb"]
    ACa, ACb = Aa @ Ca, Ab @ Cb
    if hp.get("fair", True):
        ca = wa / ((Ba.T @ Ba) * (ACa @ Aa.T)).sum().clamp_min(1e-30).sqrt()
        cb = wb / ((Bb.T @ Bb) * (ACb @ Ab.T)).sum().clamp_min(1e-30).sqrt()
    else:
        ca, cb = wa, wb
    lam, E = torch.linalg.eigh(((ca * Ca + cb * Cb) + (ca * Ca + cb * Cb).T) / 2)
    lam = lam.clamp_min(1e-8 * lam.max().clamp_min(1e-30))
    L = torch.cat([ca * Ba, cb * Bb], 1)
    RE = torch.cat([ACa, ACb], 0) @ E
    if not hp.get("whiten", True):
        return trunc_lr(L, (RE / lam) @ E.T)                    # X* = N C^{-1}
    U, S, V = lr_svd(L, (RE / lam.sqrt()) @ E.T)               # Z* = N C^{-1/2}
    tot, kept = float((S ** 2).sum()), float((S[:R] ** 2).sum())
    k = min(R, S.numel()); s = S[:k].sqrt()
    B = U[:, :k] * s
    A = (s[:, None] * V[:, :k].T) @ ((E / lam.sqrt()) @ E.T)   # X = Z_r C^{-1/2}
    if k < R:
        B = torch.cat([B, B.new_zeros(B.shape[0], R - k)], 1); A = torch.cat([A, A.new_zeros(R - k, A.shape[1])], 0)
    return B, A, tot - kept, tot


# ---------- registry ----------
METHODS = OrderedDict()


def _reg(name, fn, group, family, data, grid=None, act=False, sym=True, desc=""):
    METHODS[name] = dict(fn=fn, group=group, family=family, data=data, grid=grid or [{}], act=act, sym=sym, desc=desc)


_reg("kmerge", mm_kmerge, "prior", "A averaging", "none", desc="K-Merge Eq.8 on LoRA A,B")
_reg("delta_avg", mm_delta_avg, "prior", "A averaging", "none", desc="average B.A, SVD to rank r")
_reg("task_arith", mm_task_arith, "prior", "A scaling", "labeled val", grid=[{"lam": v} for v in (0.5, 0.75, 1.0)])
_reg("slerp", mm_slerp, "prior", "A averaging", "none")
_reg("ties", mm_ties, "prior", "B sparsify/sign", "labeled val", grid=[{"keep": v} for v in (0.1, 0.2, 0.5)])
_reg("dare", mm_dare, "prior", "B sparsify/sign", "labeled val", grid=[{"drop": v} for v in (0.5, 0.9)], sym=False)
_reg("magmax", mm_magmax, "prior", "B sparsify/sign", "none")
_reg("knots_ties", mm_knots_ties, "prior", "C subspace", "labeled val", grid=[{"keep": v} for v in (0.2, 0.5)])
_reg("tsv", mm_tsv, "prior", "C subspace", "none")
_reg("iso_c", mm_iso_c, "prior", "C subspace", "labeled val", grid=[{"alpha": v} for v in (0.5, 1.0, 1.5)])
_reg("layer_sel", mm_layer_sel, "lab v1", "H layer-aware", "labeled val", grid=[{"th": v} for v in (0.2, 0.3, 0.4)], sym=False)
_reg("ocp", mm_ocp, "lab v1", "H layer-aware", "labeled val", grid=[{"th": v} for v in (0.2, 0.3, 0.4)], sym=False)
_reg("casc_soft", mm_casc_soft, "lab v1", "H layer-aware", "none", sym=False)
# functional family. act=None: data-free from weights; "G": pseudo-inputs written by the base model from the task
# description (data-free); "U": 32 real unlabeled user inputs.
_reg("selfgram_cf", mm_gram_self, "prior", "D functional", "none", grid=[{"rho": 0.0, "fair": False, "whiten": False}],
     desc="WUDI's data-free objective (task vector as its own input proxy), solved in closed form")
_reg("regmean_cf", mm_gram_act, "prior", "D functional", "32 unlabeled", act="U", grid=[{"fair": False, "whiten": False}],
     desc="RegMean/LoRM objective, closed form, plain SVD truncation")
_reg("selfgram_ridge", mm_gram_self, "GRAM", "D functional", "none", grid=[{"rho": 0.5, "fair": True, "whiten": True}],
     desc="ablation: self-Gram + isotropic prior + balance + rank-aware (shows the self proxy itself is the problem)")
_reg("gram_iso", mm_gram_self, "GRAM", "D functional", "none", grid=[{"rho": 1.0, "fair": True, "whiten": True}],
     desc="isotropic inputs: norm-balanced average (data-free)")
_reg("regmeanG_cf", mm_gram_act, "GRAM", "D functional", "task description", act="G", grid=[{"fair": False, "whiten": False}])
_reg("gramG_nofair", mm_gram_act, "GRAM", "D functional", "task description", act="G", grid=[{"fair": False, "whiten": True}])
_reg("gramG", mm_gram_act, "GRAM", "NEW", "task description", act="G", grid=[{"fair": True, "whiten": True}],
     desc="GRAM-G: generated pseudo-inputs + balance + exact rank-r (no user data)")
_reg("gramU_nofair", mm_gram_act, "GRAM", "D functional", "32 unlabeled", act="U", grid=[{"fair": False, "whiten": True}])
_reg("gramU", mm_gram_act, "GRAM", "NEW", "32 unlabeled", act="U", grid=[{"fair": True, "whiten": True}],
     desc="GRAM-U: 32 unlabeled inputs + balance + exact rank-r")
METHOD_LIST = MODE["methods"] or list(METHODS)
SEQ_HP = {"ties": {"keep": 0.2}}


def hp_str(hp):
    return ",".join(f"{k}={v}" for k, v in sorted(hp.items())) or "-"


def _mean_gram(gram_list, mk):
    if gram_list is None:
        return None
    G = gram_list[0][mk].to(DEVICE)
    for g in gram_list[1:]:
        G = G + g[mk].to(DEVICE)
    return G / len(gram_list)


class MeanGrams:
    """Gram of a merged slot = average of its member tasks' Grams (computed per module on demand)."""
    def __init__(self, gram_list):
        self.gram_list = gram_list

    def __getitem__(self, mk):
        return _mean_gram(self.gram_list, mk)


def merge_adapters(method, fa, fb, wa=1, wb=1, hp=None, grams_a=None, grams_b=None, seed=0):
    """Merge two adapters with `method`. grams_* = list of per-leaf activation-Gram dicts (only for act methods)."""
    spec = METHODS[method]
    hp = dict(spec["grid"][0] if hp is None else hp)
    out, lost, tot = OrderedDict(), 0.0, 0.0
    for i, mk in enumerate(fa):
        ctx = {"seed": seed * 100003 + i}
        if spec["act"]:
            ctx["Ca"], ctx["Cb"] = _mean_gram(grams_a, mk), _mean_gram(grams_b, mk)
        B, A, l, t = spec["fn"](fa[mk], fb[mk], wa, wb, hp, ctx)
        out[mk] = (B, A); lost += l; tot += t
    return out, {"trunc_loss": lost / tot if tot > 0 else 0.0}


# ---------- distortion predictors and cosine ----------
def dist_self(m, t):
    """|| (X - dW_t) dW_t^T ||^2 and || dW_t dW_t^T ||^2 (data-free functional distortion of task t)."""
    (Bm, Am), (Bt, At) = m, t
    L = torch.cat([Bm, -Bt], 1)
    K = torch.cat([Am, At], 0) @ At.T
    BtB, AA = Bt.T @ Bt, At @ At.T
    return float(torch.trace(K.T @ (L.T @ L) @ K @ BtB)), float(torch.trace(AA @ BtB @ AA @ BtB))


def dist_act(m, t, C):
    """|| (X - dW_t) C^{1/2} ||^2 and || dW_t C^{1/2} ||^2 (functional distortion on inputs with Gram C)."""
    (Bm, Am), (Bt, At) = m, t
    C = C.to(DEVICE)
    L, Rt = torch.cat([Bm, -Bt], 1), torch.cat([Am, At], 0)
    return float(((L.T @ L) * (Rt @ C @ Rt.T)).sum()), float(((Bt.T @ Bt) * (At @ C @ At.T)).sum())


def rel_distortion(fm, ft, grams=None):
    num = den = 0.0
    for mk in fm:
        n_, d_ = dist_act(fm[mk], ft[mk], grams[mk]) if grams is not None else dist_self(fm[mk], ft[mk])
        num += n_; den += d_
    return num / max(den, 1e-30)


def kmerge_cos(fa, fb):
    """K-Merge similarity: layer-averaged cosine of flattened delta W."""
    cs = [float(ip(fa[mk], fb[mk]) / (ip(fa[mk], fa[mk]).sqrt() * ip(fb[mk], fb[mk]).sqrt()).clamp_min(1e-30))
          for mk in fa]
    return float(np.mean(cs))


def layer_of(mk):
    return int(re.search(r"layers\.(\d+)\.", mk).group(1))


# ============================================================================
# 4b. K-SHARE: task-conditional slots instead of merging
#     K-Merge already routes every input to the slot of its (known) task, so a slot does not need ONE weight set
#     for all its tasks. It stores a shared basis (U, V: exactly the size of one LoRA) plus a tiny r x r core per
#     task. Task t runs with U G_t V^T, an ordinary rank-r LoRA: no averaging, no cross-task interference.
# ============================================================================
def _energy(a):
    return float(((a[0].T @ a[0]) * (a[1] @ a[1].T)).sum())


def _top_left(M, k):
    U, _, _ = torch.linalg.svd(M, full_matrices=False)
    return U[:, :k]


def _psd_sqrt(M):
    e, Q = torch.linalg.eigh((M + M.T) / 2)
    return (Q * e.clamp_min(0).sqrt()) @ Q.T


def _pad_rank(B, A, r=R):
    k = B.shape[1]
    if k < r:
        B = torch.cat([B, B.new_zeros(B.shape[0], r - k)], 1)
        A = torch.cat([A, A.new_zeros(r - k, A.shape[1])], 0)
    return B, A


def kshare_module(members, rank=R, iters=6):
    """members: list of (B, A), one per task sharing the slot. Shared orthonormal U (d_out x s), V (d_in x s) found
    by HOOI on relative-energy-weighted updates (so a big adapter cannot take the whole basis); core G_t = U^T dW_t V.
    Returns per-task rank-R adapters (U G_t, V^T) and the fraction of each task's energy kept."""
    s = min(rank, members[0][0].shape[0], members[0][1].shape[1])
    en = [_energy(m) for m in members]
    c = [1.0 / max(e_, 1e-30) for e_ in en]
    U = _top_left(torch.cat([ci ** 0.5 * B @ _psd_sqrt(A @ A.T) for ci, (B, A) in zip(c, members)], 1), s)
    V = _top_left(torch.cat([ci ** 0.5 * A.T @ _psd_sqrt(B.T @ B) for ci, (B, A) in zip(c, members)], 1), s)
    for _ in range(iters):
        U = _top_left(torch.cat([ci ** 0.5 * B @ (A @ V) for ci, (B, A) in zip(c, members)], 1), s)
        V = _top_left(torch.cat([ci ** 0.5 * A.T @ (B.T @ U) for ci, (B, A) in zip(c, members)], 1), s)
    outs, kept = [], []
    for (B, A), e_ in zip(members, en):
        G = (U.T @ B) @ (A @ V)
        outs.append(_pad_rank(U @ G, V.T))
        kept.append(float((G ** 2).sum()) / max(e_, 1e-30))
    return outs, kept


def split_module(members, rank=R):
    """Simple split (the TSV-C idea): each task keeps only its own top singular directions; the rank budget is
    divided between the tasks. Also task-conditional, also exactly one slot of storage, but nothing is shared."""
    M = len(members)
    ks = [rank // M + (1 if i < rank % M else 0) for i in range(M)]
    outs, kept = [], []
    for (B, A), k in zip(members, ks):
        U, S, V = lr_svd(B, A)
        k = min(k, S.numel())
        outs.append(_pad_rank(U[:, :k] * S[:k], V[:, :k].T))
        kept.append(float((S[:k] ** 2).sum() / (S ** 2).sum().clamp_min(1e-30)))
    return outs, kept


SHARE_METHODS = OrderedDict([
    ("split", dict(fn=lambda ms: split_module(ms, R), family="task-conditional", data="none",
                   storage="= 1 slot", desc="each task keeps its own top-r/M directions (TSV-C idea)")),
    ("kshare", dict(fn=lambda ms: kshare_module(ms, R), family="NEW", data="none",
                    storage="1 slot + r x r per task", desc="K-Share: shared basis + per-task core")),
    ("kshare_eq", dict(fn=lambda ms: kshare_module(ms, R - 1), family="NEW", data="none",
                       storage="<= 1 slot", desc="K-Share with rank r-1 so basis + cores fit inside one slot")),
])


def share_adapters(name, facs):
    """Apply a K-Share-family method to a list of task adapters; returns one adapter per task + mean kept energy."""
    out = [OrderedDict() for _ in facs]
    kept_all = []
    for mk in facs[0]:
        outs, kept = SHARE_METHODS[name]["fn"]([f[mk] for f in facs])
        for i, o in enumerate(outs):
            out[i][mk] = o
        kept_all += kept
    return out, float(np.mean(kept_all))


def core_overhead_fraction():
    """Extra storage of ONE task's cores relative to one LoRA adapter."""
    lora = sum(m.lora_A["default"].weight.numel() + m.lora_B["default"].weight.numel() for m in MOD_MAP.values())
    return len(MOD_MAP) * R * R / lora

In [ ]:
# ============================================================================
# 5. SELF-TEST OF THE MERGE MATH (seconds; must pass before any experiment)
# ============================================================================
def merge_selftest():
    g = torch.Generator().manual_seed(0)
    m_, n_ = 40, 36

    def rnd(scale=1.0):
        return ((torch.randn(m_, R, generator=g) * scale).to(DEVICE),
                (torch.randn(R, n_, generator=g) / math.sqrt(n_)).to(DEVICE))

    a, b = rnd(1.0), rnd(2.5)                     # deliberately unequal norms
    Ca = torch.randn(n_, 3 * n_, generator=g); Ca = prep_gram((Ca @ Ca.T / (3 * n_)).to(DEVICE))
    Cb = torch.randn(n_, 3 * n_, generator=g); Cb = prep_gram((Cb @ Cb.T / (3 * n_)).to(DEVICE))
    checks = []

    U, S, V = lr_svd(*_cat(a, b, 1.0, 1.0))
    M = _dense(a) + _dense(b)
    checks.append(("low-rank SVD is exact", float((U * S @ V.T - M).norm() / M.norm()) < 1e-4))

    outs = {}
    for name, spec in METHODS.items():
        ctx = {"seed": 1, "Ca": Ca, "Cb": Cb}
        B, A, _, _ = spec["fn"](a, b, 1, 1, dict(spec["grid"][0]), ctx)
        outs[name] = (B, A)
        ok = B.shape == (m_, R) and A.shape == (R, n_) and bool(torch.isfinite(B).all() and torch.isfinite(A).all())
        checks.append((f"{name}: rank-{R} output, finite", ok))

    def obj_self(X, fair, rho=0.5):
        tot = 0.0
        for t in (a, b):
            Wt = _dense(t)
            Ct = (1 - rho) * (Wt.T @ Wt) / float(Wt.norm() ** 2) + (rho / n_) * torch.eye(n_, device=DEVICE)
            c = 1.0 / math.sqrt(float(torch.trace(Wt @ Ct @ Wt.T))) if fair else 1.0
            tot += c * float(torch.trace((X - Wt) @ Ct @ (X - Wt).T))
        return tot

    def obj_act(X, fair):
        tot = 0.0
        for t, C in ((a, Ca), (b, Cb)):
            Wt = _dense(t)
            c = 1.0 / math.sqrt(float(torch.trace(Wt @ C @ Wt.T))) if fair else 1.0
            tot += c * float(torch.trace((X - Wt) @ C @ (X - Wt).T))
        return tot

    for nm in ("selfgram_ridge", "gram_iso", "gramG_nofair", "gramG", "gramU_nofair", "gramU"):
        hp_ = METHODS[nm]["grid"][0]
        if METHODS[nm]["act"]:
            objf = (lambda X, f=hp_["fair"]: obj_act(X, f))
        else:
            objf = (lambda X, f=hp_["fair"], r_=hp_["rho"]: obj_self(X, f, r_))
        best = objf(_dense(outs[nm]))
        others = [objf(_dense(outs[o])) for o in outs if o != nm]
        checks.append((f"{nm} has the lowest objective among all {len(others) + 1} rank-{R} merges",
                       best <= min(others) * (1 + 1e-4)))
        Bm, Am = outs[nm]
        worse = 0
        for _ in range(30):
            Bp = Bm + 1e-2 * Bm.norm() / math.sqrt(Bm.numel()) * torch.randn(Bm.shape, generator=g).to(DEVICE)
            Ap = Am + 1e-2 * Am.norm() / math.sqrt(Am.numel()) * torch.randn(Am.shape, generator=g).to(DEVICE)
            worse += objf(Bp @ Ap) >= best * (1 - 1e-6)
        checks.append((f"{nm} is a local optimum (30/30 random rank-{R} perturbations are worse)", worse == 30))

    # balance: two non-overlapping rank-r/2 tasks of different size (together they fit the rank budget exactly):
    # balanced weights must make the relative damage of both tasks equal
    h = R // 2
    Ao = torch.linalg.qr(torch.randn(n_, R, generator=g))[0].T.to(DEVICE)
    zB, zA = torch.zeros(m_, R - h, device=DEVICE), torch.zeros(R - h, n_, device=DEVICE)
    a2 = (torch.cat([a[0][:, :h], zB], 1), torch.cat([Ao[:h], zA], 0))
    b2 = (torch.cat([b[0][:, :h], zB], 1), torch.cat([Ao[h:2 * h], zA], 0))
    Bm, Am = METHODS["gram_iso"]["fn"](a2, b2, 1, 1, dict(METHODS["gram_iso"]["grid"][0]), {})[:2]
    ra = float((Bm @ Am - _dense(a2)).norm() / _dense(a2).norm())
    rb = float((Bm @ Am - _dense(b2)).norm() / _dense(b2).norm())
    checks.append(("balanced weights equalise relative damage of non-overlapping tasks", abs(ra - rb) < 1e-3 * max(ra, rb)))

    for nm in ("kmerge", "delta_avg", "tsv", "gram_iso", "gramG", "gramU"):
        ctx_ab = {"seed": 1, "Ca": Ca, "Cb": Cb}; ctx_ba = {"seed": 1, "Ca": Cb, "Cb": Ca}
        Xab = _dense(METHODS[nm]["fn"](a, b, 1, 1, dict(METHODS[nm]["grid"][0]), ctx_ab)[:2])
        Xba = _dense(METHODS[nm]["fn"](b, a, 1, 1, dict(METHODS[nm]["grid"][0]), ctx_ba)[:2])
        checks.append((f"{nm} is symmetric (merge(a,b) == merge(b,a))", float((Xab - Xba).norm() / Xab.norm()) < 1e-3))

    W1 = _dense(METHODS["delta_avg"]["fn"](a, a, 1, 1, {}, {})[:2])
    checks.append(("merging an adapter with itself returns it (delta_avg)", float((W1 - _dense(a)).norm() / _dense(a).norm()) < 1e-4))
    W2 = _dense(METHODS["gramU"]["fn"](a, a, 1, 1, dict(METHODS["gramU"]["grid"][0]), {"Ca": Ca, "Cb": Ca})[:2])
    checks.append(("merging an adapter with itself returns it (gramU)", float((W2 - _dense(a)).norm() / _dense(a).norm()) < 1e-3))

    # ---- K-Share checks ----
    (o1,), k1 = kshare_module([a])
    checks.append(("K-Share with one task returns that task's adapter",
                   float((_dense(o1) - _dense(a)).norm() / _dense(a).norm()) < 1e-4 and abs(k1[0] - 1) < 1e-4))
    outs_k, kept_k = kshare_module([a, b])
    outs_s, kept_s = split_module([a, b])
    checks.append(("K-Share keeps at least as much of each task as the simple split (sum of kept fractions)",
                   sum(kept_k) >= sum(kept_s) - 1e-3))
    checks.append(("K-Share / split outputs are finite rank-R adapters",
                   all(o[0].shape == (m_, R) and o[1].shape == (R, n_) and bool(torch.isfinite(o[0]).all())
                       for o in outs_k + outs_s)))
    outs_e, _ = kshare_module([a, b], rank=R - 1)
    checks.append(("kshare_eq really uses rank R-1 (basis + cores fit inside one slot)",
                   int(torch.linalg.matrix_rank(_dense(outs_e[0]))) <= R - 1))
    outs_3, kept_3 = kshare_module([a, b, rnd(0.7)])
    checks.append(("K-Share works with 3 tasks in one slot", len(outs_3) == 3 and all(0 <= x <= 1 + 1e-6 for x in kept_3)))

    n_ok = sum(ok for _, ok in checks)
    for nm, ok in checks:
        if not ok:
            print("   FAILED:", nm)
    log(f"MERGE MATH SELF-TEST: {n_ok}/{len(checks)} checks passed")
    assert n_ok == len(checks), "merge self-test failed: do not trust results"


merge_selftest()

In [ ]:
# ============================================================================
# 6. TRAIN ADAPTERS, REFERENCE SCORES, ACTIVATION GRAMS
# ============================================================================
FAC = {}
for seed in SEEDS:
    FAC[seed] = {}
    for lang in LANGS:
        FAC[seed][lang] = state_to_fac(train_adapter(lang, seed))

# Grams live on the GPU when they fit comfortably, otherwise in CPU RAM (moved per module when used)
_gram_bytes = 2 * len(LANGS) * sum(m.lora_A["default"].weight.shape[1] ** 2 * 4 for m in MOD_MAP.values())
GRAM_DEV = DEVICE if (DEVICE.type == "cuda" and
                      _gram_bytes < 0.25 * torch.cuda.get_device_properties(0).total_memory) else torch.device("cpu")

GEN_PATH = os.path.join(RUN_DIR, "generated_pseudo_inputs.csv")
GEN_SENTS = {}
if os.path.exists(GEN_PATH):
    _g = pd.read_csv(GEN_PATH, keep_default_na=False)
    GEN_SENTS = {l: list(_g[_g.lang == l].sentence) for l in LANGS}
for lang in LANGS:
    if len(GEN_SENTS.get(lang, [])) < N_BUFFER:
        GEN_SENTS[lang] = generate_pseudo_inputs(lang)
pd.DataFrame([{"lang": l, "sentence": s} for l in LANGS for s in GEN_SENTS[l]]).to_csv(GEN_PATH, index=False)
for lang in LANGS:
    log(f"GRAM-G pseudo-inputs for {lang} (base model, task description only): " + " | ".join(GEN_SENTS[lang][:3]))

GRAMS, GRAMS_GEN = {}, {}
for lang in LANGS:
    raw = collect_grams([r["src"] for r in DATA[lang]["train"][:N_BUFFER]])      # unlabeled real inputs
    GRAMS[lang] = OrderedDict((mk, prep_gram(raw[mk]).to(GRAM_DEV)) for mk in MOD_MAP)
    raw = collect_grams(GEN_SENTS[lang])                                          # generated pseudo-inputs
    GRAMS_GEN[lang] = OrderedDict((mk, prep_gram(raw[mk]).to(GRAM_DEV)) for mk in MOD_MAP)
    del raw
GRAMSET = {"U": GRAMS, "G": GRAMS_GEN}
log(f"activation Grams ready ({N_BUFFER} real unlabeled + {N_BUFFER} generated inputs per language; stored on {GRAM_DEV})")


def solo_key(seed, lang):
    return f"solo|{seed}|{lang}"


REF_ROWS = []
for lang in LANGS:
    mb = evaluate("base", None, lang, "test")
    for seed in SEEDS:
        ms = evaluate(solo_key(seed, lang), FAC[seed][lang], lang, "test")
        evaluate(solo_key(seed, lang), FAC[seed][lang], lang, "val")
        REF_ROWS.append({"lang": lang, "seed": seed, "base_em": mb["em"], "solo_em": ms["em"], "solo_f1": ms["f1"],
                         "solo_gleu": ms["gleu"], "norm_dW": math.sqrt(sum(float(ip(f, f)) for f in FAC[seed][lang].values()))})
REF = pd.DataFrame(REF_ROWS)
REF.to_csv(os.path.join(RUN_DIR, "adapter_quality.csv"), index=False)
print(REF.round(3).to_string(index=False))
for _, r in REF.iterrows():
    if r.solo_em - r.base_em < 0.10:
        print(f"   NOTE {r.lang} seed {r.seed}: adapter barely beats the base model -> consider more EPOCHS/higher LR")
    if r.solo_em > 0.98:
        print(f"   NOTE {r.lang} seed {r.seed}: adapter is saturated (EM>0.98) -> positive transfer cannot show")

In [ ]:
# ============================================================================
# 7. PAIRWISE BENCHMARK: every method x every pair x every seed
#    hyper-parameters are tuned on VAL, results reported on TEST
# ============================================================================
PAIR_CSV = os.path.join(RUN_DIR, "pairwise_results.csv")
LAYER_CSV = os.path.join(RUN_DIR, "layer_diagnostics.csv")
VALSEL_CSV = os.path.join(RUN_DIR, "val_selection.csv")
DONE = set()
if os.path.exists(PAIR_CSV):
    _d = pd.read_csv(PAIR_CSV)
    DONE = set(zip(_d.seed, _d.pair, _d.method))


def _append(path, rows):
    """Append rows to a CSV; if the columns changed (e.g. after a code update) the file is rewritten safely."""
    if not rows:
        return
    df = pd.DataFrame(rows)
    if not os.path.exists(path):
        df.to_csv(path, index=False)
        return
    cols = list(pd.read_csv(path, nrows=0).columns)
    if set(df.columns) <= set(cols):
        df.reindex(columns=cols).to_csv(path, mode="a", header=False, index=False)
    else:
        pd.concat([pd.read_csv(path), df], ignore_index=True).to_csv(path, index=False)


def tkey(t):
    return t if isinstance(t, str) else "(" + tkey(t[0]) + "+" + tkey(t[1]) + ")"


def canon(x, y):
    return (x, y) if tkey(x) <= tkey(y) else (y, x)


def leaves(t):
    return [t] if isinstance(t, str) else leaves(t[0]) + leaves(t[1])


def merged_key(seed, method, hp, a, b):
    tree = canon(a, b) if METHODS[method]["sym"] else (a, b)
    return f"m|{seed}|{method}|{hp_str(hp)}|{tkey(tree)}"


def nz(x, s):
    """Normalised score = merged / own adapter. If the own adapter is very weak (EM < 0.2) the ratio is
    meaningless, so the raw score is used (the adapter-quality table flags such adapters)."""
    return x / s if s >= 0.2 else x


def solo_em(seed, lang, split):
    return EVAL_CACHE[f"{solo_key(seed, lang)}|{lang}|{split}"]["em"]


def pair_diagnostics(seed, a, b):
    fa, fb = FAC[seed][a], FAC[seed][b]
    fk, _ = merge_adapters("kmerge", fa, fb)
    rows = []
    for mk in fa:
        sa, da = dist_self(fk[mk], fa[mk]); sb, db = dist_self(fk[mk], fb[mk])
        xa, ya = dist_act(fk[mk], fa[mk], GRAMS[a][mk]); xb, yb = dist_act(fk[mk], fb[mk], GRAMS[b][mk])
        ga_, ha_ = dist_act(fk[mk], fa[mk], GRAMS_GEN[a][mk]); gb_, hb_ = dist_act(fk[mk], fb[mk], GRAMS_GEN[b][mk])
        rows.append({"seed": seed, "pair": f"{a}-{b}", "layer": layer_of(mk), "module": mk.split(".")[-1],
                     "cos": float(ip(fa[mk], fb[mk]) / (ip(fa[mk], fa[mk]).sqrt() * ip(fb[mk], fb[mk]).sqrt())),
                     "subspace_overlap": overlap(fa[mk], fb[mk]),
                     "dself_a": sa / max(da, 1e-30), "dself_b": sb / max(db, 1e-30),
                     "dact_a": xa / max(ya, 1e-30), "dact_b": xb / max(yb, 1e-30),
                     "dgen_a": ga_ / max(ha_, 1e-30), "dgen_b": gb_ / max(hb_, 1e-30)})
    return rows


def _record_pair(seed, a, b, method, group, family, data, hp, cos_ab, fma, fmb, ta, tb, msec, trunc):
    """One results row. fma / fmb = the adapter task a / task b actually runs with (same for a merge)."""
    fa, fb = FAC[seed][a], FAC[seed][b]
    row = {"seed": seed, "pair": f"{a}-{b}", "a": a, "b": b, "method": method, "group": group,
           "family": family, "data": data, "hp": hp, "cos": cos_ab,
           "norm_a": REF.query("lang == @a and seed == @seed").norm_dW.iloc[0],
           "norm_b": REF.query("lang == @b and seed == @seed").norm_dW.iloc[0],
           "solo_a": solo_em(seed, a, "test"), "solo_b": solo_em(seed, b, "test"),
           "dself_a": rel_distortion(fma, fa), "dself_b": rel_distortion(fmb, fb),
           "dact_a": rel_distortion(fma, fa, GRAMS[a]), "dact_b": rel_distortion(fmb, fb, GRAMS[b]),
           "dgen_a": rel_distortion(fma, fa, GRAMS_GEN[a]), "dgen_b": rel_distortion(fmb, fb, GRAMS_GEN[b]),
           "merge_ms": msec, "trunc_loss": trunc}
    for side, mt in (("a", ta), ("b", tb)):
        for k_, v_ in mt.items():
            row[f"{k_}_{side}"] = v_
    row["nem_a"], row["nem_b"] = nz(ta["em"], row["solo_a"]), nz(tb["em"], row["solo_b"])
    _append(PAIR_CSV, [row])
    DONE.add((seed, f"{a}-{b}", method))
    log(f"seed {seed} {a}-{b:3s} {method:13s} EM {a}={ta['em']:.2f} {b}={tb['em']:.2f} | "
        f"normEM {row['nem_a']:.2f}/{row['nem_b']:.2f} | hp {hp}")


for seed in SEEDS:
    for (a, b) in PAIRS:
        pair = f"{a}-{b}"
        if not os.path.exists(LAYER_CSV) or pair not in set(
                pd.read_csv(LAYER_CSV).query("seed == @seed").pair):
            _append(LAYER_CSV, pair_diagnostics(seed, a, b))
        fa, fb = FAC[seed][a], FAC[seed][b]
        cos_ab = kmerge_cos(fa, fb)
        for method in METHOD_LIST:
            if (seed, pair, method) in DONE:
                continue
            spec = METHODS[method]
            ga, gb = ([GRAMSET[spec["act"]][a]], [GRAMSET[spec["act"]][b]]) if spec["act"] else (None, None)
            best_hp, val_rows = spec["grid"][0], []
            if len(spec["grid"]) > 1:
                best_score = -1e9
                for hp in spec["grid"]:
                    key = merged_key(seed, method, hp, a, b)
                    lazy = (lambda hp=hp: merge_adapters(method, fa, fb, hp=hp, grams_a=ga, grams_b=gb, seed=seed)[0])
                    va, vb = evaluate(key, lazy, a, "val"), evaluate(key, lazy, b, "val")
                    sc = (nz(va["em"], solo_em(seed, a, "val")) + nz(vb["em"], solo_em(seed, b, "val"))) / 2 \
                        + 1e-3 * (va["f1"] + vb["f1"])
                    val_rows.append({"seed": seed, "pair": pair, "method": method, "hp": hp_str(hp), "val_score": sc})
                    if sc > best_score:
                        best_score, best_hp = sc, hp
                _append(VALSEL_CSV, val_rows)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            fm, info = merge_adapters(method, fa, fb, hp=best_hp, grams_a=ga, grams_b=gb, seed=seed)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            msec = (time.time() - t0) * 1000
            key = merged_key(seed, method, best_hp, a, b)
            ta, tb = evaluate(key, fm, a, "test"), evaluate(key, fm, b, "test")
            _record_pair(seed, a, b, method, spec["group"], spec["family"], spec["data"], hp_str(best_hp), cos_ab,
                         fm, fm, ta, tb, msec, info["trunc_loss"])
        # ---- K-Share family: one slot, but each task decodes its own adapter (task identity is known) ----
        for sname in MODE["share_methods"]:
            if (seed, pair, sname) in DONE:
                continue
            spec = SHARE_METHODS[sname]
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            (fa2, fb2), kept = share_adapters(sname, [fa, fb])
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            msec = (time.time() - t0) * 1000
            base_key = f"s|{seed}|{sname}|{tkey(canon(a, b))}"
            ta, tb = evaluate(base_key + f"@{a}", fa2, a, "test"), evaluate(base_key + f"@{b}", fb2, b, "test")
            _record_pair(seed, a, b, sname, "K-Share", spec["family"], spec["data"], "-", cos_ab,
                         fa2, fb2, ta, tb, msec, 1.0 - kept)
log("pairwise benchmark done")

In [ ]:
# ============================================================================
# 8. ONLINE K-MERGE: adapters arrive one at a time, K slots, every arrival order
#    policies: kmerge (new -> most cosine-similar slot), global_cos (best cosine pair among slots+new),
#              global_dist (GRAM-Select: lowest predicted distortion on generated pseudo-inputs, data-free),
#              oracle (labeled val, upper bound)
# ============================================================================
SEQ_CSV = os.path.join(RUN_DIR, "sequence_results.csv")
SEQ_DONE = set()
if os.path.exists(SEQ_CSV):
    _d = pd.read_csv(SEQ_CSV)
    SEQ_DONE = set(zip(_d["set"], _d["seed"], _d["rule"], _d["policy"], _d["order"]))
SEQ_FAC = {}


def seq_hp(rule):
    return SEQ_HP.get(rule, METHODS[rule]["grid"][0])


def seq_fac(seed, rule, tree):
    if isinstance(tree, str):
        return FAC[seed][tree]
    ck = (seed, rule, tkey(tree))
    if ck not in SEQ_FAC:
        x, y = tree
        act = METHODS[rule]["act"]
        SEQ_FAC[ck] = merge_adapters(rule, seq_fac(seed, rule, x), seq_fac(seed, rule, y),
                                     wa=len(leaves(x)), wb=len(leaves(y)), hp=seq_hp(rule),
                                     grams_a=[GRAMSET[act][l] for l in leaves(x)] if act else None,
                                     grams_b=[GRAMSET[act][l] for l in leaves(y)] if act else None, seed=seed)[0]
    return SEQ_FAC[ck]


def seq_eval(seed, rule, tree, lang, split):
    if isinstance(tree, str):
        return evaluate(solo_key(seed, tree), FAC[seed][tree], lang, split)
    return evaluate(f"m|{seed}|{rule}|{hp_str(seq_hp(rule))}|{tkey(tree)}",
                    lambda: seq_fac(seed, rule, tree), lang, split)


def config_score(seed, rule, slots, split):
    vals = [nz(seq_eval(seed, rule, s, l, split)["em"], solo_em(seed, l, split)) for s in slots for l in leaves(s)]
    return min(vals), float(np.mean(vals))


def run_sequence(seed, rule, policy, order):
    slots = []
    for lang in order:
        if len(slots) < K_SLOTS:
            slots.append(lang); continue
        items = slots + [lang]
        if policy == "kmerge":
            cands = [(i, len(items) - 1) for i in range(len(slots))]
        else:
            cands = list(itertools.combinations(range(len(items)), 2))

        def merged_slots(i, j):
            return [items[k] for k in range(len(items)) if k not in (i, j)] + [canon(items[i], items[j])]

        if policy in ("kmerge", "global_cos"):
            scores = [kmerge_cos(seq_fac(seed, rule, items[i]), seq_fac(seed, rule, items[j])) for i, j in cands]
            i, j = cands[int(np.argmax(scores))]
        elif policy == "global_dist":
            scores = []
            for i, j in cands:
                x, y = items[i], items[j]
                fm = seq_fac(seed, rule, canon(x, y))
                nx_, ny_ = len(leaves(x)), len(leaves(y))
                gx, gy = MeanGrams([GRAMS_GEN[l] for l in leaves(x)]), MeanGrams([GRAMS_GEN[l] for l in leaves(y)])
                scores.append((nx_ * rel_distortion(fm, seq_fac(seed, rule, x), gx) +
                               ny_ * rel_distortion(fm, seq_fac(seed, rule, y), gy)) / (nx_ + ny_))
            i, j = cands[int(np.argmin(scores))]
        elif policy == "oracle":
            scores = [config_score(seed, rule, merged_slots(i, j), "val") for i, j in cands]
            i, j = cands[max(range(len(cands)), key=lambda c: scores[c])]
        slots = merged_slots(i, j)
    return slots


for seed in SEEDS:
    for rule in MODE["seq_rules"]:
        SEQ_FAC.clear()                      # merged trees never cross seeds/rules: keep GPU memory small
        for set_name, set_langs in MODE["seq_sets"].items():
            orders = list(itertools.permutations(set_langs))
            if MODE["seq_orders_max"]:
                orders = orders[:MODE["seq_orders_max"]]
            for policy in MODE["seq_policies"]:
                rows = []
                for order in orders:
                    okey = ">".join(order)
                    if (set_name, seed, rule, policy, okey) in SEQ_DONE:
                        continue
                    slots = run_sequence(seed, rule, policy, order)
                    per = {l: seq_eval(seed, rule, s, l, "test") for s in slots for l in leaves(s)}
                    nems = {l: nz(per[l]["em"], solo_em(seed, l, "test")) for l in per}
                    rows.append({"set": set_name, "seed": seed, "rule": rule, "policy": policy, "order": okey,
                                 "slots": " | ".join(tkey(s) for s in slots),
                                 "avg_em": float(np.mean([per[l]["em"] for l in per])),
                                 "worst_em": float(min(per[l]["em"] for l in per)),
                                 "avg_nem": float(np.mean(list(nems.values()))),
                                 "worst_nem": float(min(nems.values())),
                                 **{f"em_{l}": (per[l]["em"] if l in per else np.nan) for l in LANGS}})
                _append(SEQ_CSV, rows)
                if rows:
                    log(f"seq {set_name} seed {seed} {rule:10s} {policy:12s} avg normEM "
                        f"{np.mean([r['avg_nem'] for r in rows]):.3f} | worst {np.mean([r['worst_nem'] for r in rows]):.3f}")
log("online sequence benchmark done (merging rules)")

# ---------------- K-Share online: slots share a basis; the task grouping can differ per LAYER ----------------
#   rule   : "kshare" (shared basis + per-task cores) or "split" (each task keeps its own top directions)
#   policy : "kmerge"      new adapter joins the most cosine-similar slot (K-Merge's rule, same grouping in all layers)
#            "global_loss" merge the pair (among stored slots + new) that loses the least energy, same in all layers
#            "layer_loss"  the same decision taken independently in every layer (uses layer-localised conflict)
LAYERS = sorted({layer_of(mk) for mk in MOD_MAP})
MODS_BY_LAYER = {l: [mk for mk in MOD_MAP if layer_of(mk) == l] for l in LAYERS}
SH_CACHE = {}


def sh_rep(seed, rule, layer, tree):
    """One slot at one layer: {task: {module: (B, A)}} + per-task lost / total energy of the merge that built it."""
    ck = (seed, rule, layer, tkey(tree))
    if ck in SH_CACHE:
        return SH_CACHE[ck]
    if isinstance(tree, str):
        out = ({tree: {mk: FAC[seed][tree][mk] for mk in MODS_BY_LAYER[layer]}}, {tree: 0.0}, {tree: 1.0})
    else:
        rx, _, _ = sh_rep(seed, rule, layer, tree[0])
        ry, _, _ = sh_rep(seed, rule, layer, tree[1])
        cur = {**rx, **ry}
        tasks = sorted(cur)
        rep = {t: {} for t in tasks}
        lostE, totE = {t: 0.0 for t in tasks}, {t: 0.0 for t in tasks}
        fn = kshare_module if rule == "kshare" else split_module
        for mk in MODS_BY_LAYER[layer]:
            outs, kept = fn([cur[t][mk] for t in tasks])
            for t, o, kp in zip(tasks, outs, kept):
                e_ = _energy(cur[t][mk])
                rep[t][mk] = o
                lostE[t] += (1 - kp) * e_
                totE[t] += e_
        out = (rep, lostE, totE)
    SH_CACHE[ck] = out
    return out


def sh_loss(seed, rule, x, y, layers):
    """Sum over the tasks of the merged slot of the relative energy each loses, over `layers` (data-free)."""
    lost, tot = defaultdict(float), defaultdict(float)
    for l in layers:
        _, lE, tE = sh_rep(seed, rule, l, canon(x, y))
        for t in lE:
            lost[t] += lE[t]
            tot[t] += tE[t]
    return sum(lost[t] / max(tot[t], 1e-30) for t in lost)


def sh_cos(seed, rule, tree, lang):
    """K-Merge similarity between a slot (mean of its members' adapters) and the incoming adapter."""
    cs = []
    for l in LAYERS:
        rep, _, _ = sh_rep(seed, rule, l, tree)
        mem = list(rep)
        for mk in MODS_BY_LAYER[l]:
            new = FAC[seed][lang][mk]
            ipn = sum(float(ip(new, rep[t][mk])) for t in mem) / len(mem)
            nn = sum(float(ip(rep[t][mk], rep[u][mk])) for t in mem for u in mem) / len(mem) ** 2
            cs.append(ipn / max(math.sqrt(max(nn, 0.0)) * math.sqrt(float(ip(new, new))), 1e-30))
    return float(np.mean(cs))


def run_share_sequence(seed, rule, policy, order):
    state = {l: [] for l in LAYERS}
    for lang in order:
        if len(state[LAYERS[0]]) < K_SLOTS:
            for l in LAYERS:
                state[l].append(lang)
            continue
        if policy == "layer_loss":
            for l in LAYERS:
                items = state[l] + [lang]
                cands = list(itertools.combinations(range(len(items)), 2))
                i, j = cands[int(np.argmin([sh_loss(seed, rule, items[i], items[j], [l]) for i, j in cands]))]
                state[l] = [items[k] for k in range(len(items)) if k not in (i, j)] + [canon(items[i], items[j])]
        else:
            items = state[LAYERS[0]] + [lang]
            if policy == "kmerge":
                cands = [(i, len(items) - 1) for i in range(len(items) - 1)]
                sc = [-sh_cos(seed, rule, items[i], lang) for i, _ in cands]
            else:
                cands = list(itertools.combinations(range(len(items)), 2))
                sc = [sh_loss(seed, rule, items[i], items[j], LAYERS) for i, j in cands]
            i, j = cands[int(np.argmin(sc))]
            new_state = [items[k] for k in range(len(items)) if k not in (i, j)] + [canon(items[i], items[j])]
            for l in LAYERS:
                state[l] = list(new_state)
    return state


def share_task_adapter(seed, rule, state, t):
    fac, path = OrderedDict(), []
    for l in LAYERS:
        tree = next(tr for tr in state[l] if t in leaves(tr))
        rep, _, _ = sh_rep(seed, rule, l, tree)
        fac.update(rep[t])
        path.append(tkey(tree))
    return fac, path


for seed in SEEDS:
    for rule in MODE["share_rules"]:
        SH_CACHE.clear()
        for set_name, set_langs in MODE["seq_sets"].items():
            orders = list(itertools.permutations(set_langs))
            if MODE["seq_orders_max"]:
                orders = orders[:MODE["seq_orders_max"]]
            for policy in MODE["share_policies"]:
                rows = []
                for order in orders:
                    okey = ">".join(order)
                    if (set_name, seed, rule, policy, okey) in SEQ_DONE:
                        continue
                    state = run_share_sequence(seed, rule, policy, order)
                    per = {}
                    for t in order:
                        fac_t, path = share_task_adapter(seed, rule, state, t)
                        body = path[0] if len(set(path)) == 1 else hashlib.md5("/".join(path).encode()).hexdigest()[:16]
                        per[t] = evaluate(f"sh|{seed}|{rule}|{body}@{t}", fac_t, t, "test")
                    groupings = Counter(" | ".join(sorted(tkey(tr) for tr in state[l])) for l in LAYERS)
                    nems = {l: nz(per[l]["em"], solo_em(seed, l, "test")) for l in per}
                    rows.append({"set": set_name, "seed": seed, "rule": rule, "policy": policy, "order": okey,
                                 "slots": (groupings.most_common(1)[0][0] if len(groupings) == 1 else
                                           f"{len(groupings)} groupings across layers; most common: "
                                           f"{groupings.most_common(1)[0][0]}"),
                                 "avg_em": float(np.mean([per[l]["em"] for l in per])),
                                 "worst_em": float(min(per[l]["em"] for l in per)),
                                 "avg_nem": float(np.mean(list(nems.values()))),
                                 "worst_nem": float(min(nems.values())),
                                 **{f"em_{l}": (per[l]["em"] if l in per else np.nan) for l in LANGS}})
                _append(SEQ_CSV, rows)
                if rows:
                    log(f"share {set_name} seed {seed} {rule:7s} {policy:12s} avg normEM "
                        f"{np.mean([r['avg_nem'] for r in rows]):.3f} | worst {np.mean([r['worst_nem'] for r in rows]):.3f}")
log("online sequence benchmark done (K-Share rules)")

In [ ]:
# ============================================================================
# 8c. SLOT CAPACITY: how many tasks can share ONE rank-r slot?
#     Every 3-language and 4-language subset is put into a single slot (K = 1). A merge keeps one adapter for all
#     of them; `split` gives each task only r/M of the directions; K-Share keeps one shared rank-r basis plus a tiny
#     r x r core per task. Pairs (M = 2) come from the pairwise benchmark above.
# ============================================================================
CAP_CSV = os.path.join(RUN_DIR, "capacity_results.csv")
CAP_DONE = set()
if os.path.exists(CAP_CSV):
    _d = pd.read_csv(CAP_CSV)
    CAP_DONE = set(zip(_d["seed"], _d["slot"], _d["method"]))
CAP_MERGE = [m_ for m_ in MODE.get("cap_merge", []) if m_ in METHODS]
CAP_SHARE = [m_ for m_ in MODE.get("cap_share", []) if m_ in SHARE_METHODS]


def cap_tree(langs):
    langs = sorted(langs)
    t = langs[0]
    for l in langs[1:]:
        t = canon(t, l)
    return t


for seed in SEEDS:
    SEQ_FAC.clear()
    for M in MODE.get("cap_sizes", []):
        rows = []
        for S in itertools.combinations(LANGS, M):
            tree = cap_tree(S)
            slot = tkey(tree)
            for rule in CAP_MERGE:
                if (seed, slot, rule) in CAP_DONE:
                    continue
                for l in S:
                    m_ = seq_eval(seed, rule, tree, l, "test")
                    rows.append({"seed": seed, "M": M, "slot": slot, "method": rule, "kind": "merge", "lang": l,
                                 "em": m_["em"], "solo_em": solo_em(seed, l, "test"),
                                 "nem": nz(m_["em"], solo_em(seed, l, "test")), "copy": m_["copy"], "leak": m_["leak"],
                                 "kept_energy": np.nan})
            for sname in CAP_SHARE:
                if (seed, slot, sname) in CAP_DONE:
                    continue
                facs_, kept_ = share_adapters(sname, [FAC[seed][l] for l in S])
                for l, f_ in zip(S, facs_):
                    m_ = evaluate(f"s|{seed}|{sname}|{slot}@{l}", f_, l, "test")
                    rows.append({"seed": seed, "M": M, "slot": slot, "method": sname, "kind": "share", "lang": l,
                                 "em": m_["em"], "solo_em": solo_em(seed, l, "test"),
                                 "nem": nz(m_["em"], solo_em(seed, l, "test")), "copy": m_["copy"], "leak": m_["leak"],
                                 "kept_energy": kept_})
        _append(CAP_CSV, rows)
        if rows:
            d_ = pd.DataFrame(rows).groupby("method").nem.mean()
            log(f"capacity seed {seed} M={M}: " + " | ".join(f"{k_} {v_:.3f}" for k_, v_ in d_.sort_values().items()))
    SEQ_FAC.clear()
log("slot-capacity benchmark done")

In [ ]:
# ============================================================================
# 9. ANALYSIS, FIGURES AND THE SUMMARY TO SEND BACK
# ============================================================================
pw = pd.read_csv(PAIR_CSV)
pw = pw[pw.method.isin(list(METHOD_LIST) + list(MODE["share_methods"])) & pw.seed.isin(SEEDS)]
pw["avg_nem"] = (pw.nem_a + pw.nem_b) / 2
pw["worst_nem"] = pw[["nem_a", "nem_b"]].min(axis=1)
pw["imbalance"] = (pw.nem_a - pw.nem_b).abs()
for k_ in ("em", "f1", "gleu", "edit", "copy", "over", "leak"):
    pw[k_] = (pw[f"{k_}_a"] + pw[f"{k_}_b"]) / 2
km = pw[pw.method == "kmerge"].set_index(["seed", "pair"]).avg_nem
pw["beats_kmerge"] = [float(r.avg_nem > km.get((r.seed, r.pair), np.nan) + 1e-9) if r.method != "kmerge" else np.nan
                      for r in pw.itertuples()]

board = (pw.groupby("method").agg(group=("group", "first"), data=("data", "first"), avg_normEM=("avg_nem", "mean"),
                                   worst_normEM=("worst_nem", "mean"), EM=("em", "mean"), F1=("f1", "mean"),
                                   GLEU=("gleu", "mean"), edit=("edit", "mean"), win_vs_kmerge=("beats_kmerge", "mean"),
                                   merge_ms=("merge_ms", "mean"), n=("avg_nem", "size"))
         .sort_values("avg_normEM", ascending=False))
fail = (pw.groupby("method").agg(copy_underfix=("copy", "mean"), overcorrect=("over", "mean"), leak=("leak", "mean"),
                                  imbalance=("imbalance", "mean"), trunc_loss=("trunc_loss", "mean"))
        .loc[board.index])


def _spearman(x, y):
    x, y = pd.Series(np.asarray(x, dtype=float)), pd.Series(np.asarray(y, dtype=float))
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 4 or x.nunique() < 2 or y.nunique() < 2:
        return float("nan")
    return float(x.rank().corr(y.rank()))


norm_dom = {}
for mth, g_ in pw.groupby("method"):
    norm_dom[mth] = _spearman(np.log(g_.norm_a / g_.norm_b), g_.nem_a - g_.nem_b)
fail["norm_dominance_rho"] = pd.Series(norm_dom)

long = pd.concat([pd.DataFrame({"method": pw.method, "pair": pw.pair, "seed": pw.seed, "cos": pw.cos,
                                "dself": pw[f"dself_{s}"], "dgen": pw[f"dgen_{s}"], "dact": pw[f"dact_{s}"],
                                "damage": 1 - pw[f"nem_{s}"]})
                  for s in ("a", "b")])
pred_rows = []
for scope, d in (("K-Merge merges only", long[long.method == "kmerge"]), ("all methods pooled", long)):
    pred_rows.append({"scope": scope, "n": len(d),
                      "cosine (K-Merge)": _spearman(-d.cos, d.damage),
                      "self-Gram distortion": _spearman(d.dself, d.damage),
                      "generated-input distortion (data-free)": _spearman(d.dgen, d.damage),
                      "unlabeled-input distortion": _spearman(d.dact, d.damage)})
pred = pd.DataFrame(pred_rows)

lay = pd.read_csv(LAYER_CSV)
lay = lay[lay.seed.isin(SEEDS)]
lay["func_conflict"] = (lay.dact_a + lay.dact_b) / 2
heat = lay.groupby(["pair", "layer"]).func_conflict.mean().unstack("layer")
top_layers = {p: ", ".join(str(int(c)) for c in heat.loc[p].sort_values(ascending=False).index[:5]) for p in heat.index}

seq_tab = None
if os.path.exists(SEQ_CSV):
    sq = pd.read_csv(SEQ_CSV)
    sq = sq[sq.seed.isin(SEEDS)]
    seq_tab = (sq.groupby(["set", "rule", "policy"])
               .agg(avg_normEM=("avg_nem", "mean"), worst_normEM=("worst_nem", "mean"),
                    worst_order_avg=("avg_nem", "min"), order_spread_std=("avg_nem", "std"), runs=("avg_nem", "size"))
               .reset_index())

cap_tab = None
if os.path.exists(CAP_CSV):
    cp = pd.read_csv(CAP_CSV)
    cp = cp[cp.seed.isin(SEEDS)]
    p2 = pd.concat([pd.DataFrame({"seed": pw.seed, "M": 2, "slot": pw.pair, "method": pw.method, "lang": pw[s_],
                                  "nem": pw[f"nem_{s_}"]}) for s_ in ("a", "b")])
    allc = pd.concat([p2[p2.method.isin(set(cp.method))], cp[["seed", "M", "slot", "method", "lang", "nem"]]],
                     ignore_index=True)
    if len(cp):
        cap_tab = pd.concat({"avg_normEM": allc.groupby(["method", "M"]).nem.mean().unstack("M"),
                             "worst_normEM": allc.groupby(["method", "M", "seed", "slot"]).nem.min()
                                                 .groupby(["method", "M"]).mean().unstack("M")}, axis=1)
        cap_tab = cap_tab.loc[[m_ for m_ in ("kmerge", "delta_avg", "regmean_cf", "gramU", "gramG", "split", "kshare",
                                             "kshare_eq") if m_ in cap_tab.index]]

# ---------------- figures (validated palette: blue / orange / aqua; one axis each) ----------------
INK, INK2, MUTED, GRID, AXIS, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
GROUP_COLORS = {"prior": "#2a78d6", "lab v1": "#2a78d6", "GRAM": "#eb6834", "K-Share": "#1baf7a"}
SEQ_BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(board) + 1.2))
yy = np.arange(len(board))[::-1]
ax.barh(yy, board.avg_normEM, height=0.6, color=[GROUP_COLORS[g] for g in board.group], zorder=2)
ax.set_yticks(yy); ax.set_yticklabels(board.index, color=INK)
if "kmerge" in board.index:
    ax.axvline(board.loc["kmerge", "avg_normEM"], color=MUTED, lw=1, ls="--", zorder=3)
    ax.text(board.loc["kmerge", "avg_normEM"], len(board) - 0.3, " K-Merge", color=INK2, va="bottom", fontsize=8)
ax.grid(axis="x", color=GRID, lw=0.8, zorder=0)
ax.set_xlabel("Normalised exact match after merging (merged EM / own-adapter EM), mean over pairs and seeds")
ax.set_title("Which method keeps both languages working in one slot?", loc="left", color=INK, fontsize=11)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=GROUP_COLORS[g_], label=lab_) for g_, lab_ in
                   (("prior", "merging: prior work + earlier ideas"), ("GRAM", "merging: GRAM (new) + ablations"),
                    ("K-Share", "no merging: K-Share (new) + split baseline"))],
          frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "fig_leaderboard.png"), dpi=150); plt.show()

from matplotlib.colors import LinearSegmentedColormap
fig, ax = plt.subplots(figsize=(min(14, max(8, 0.32 * heat.shape[1] + 2.5)), 0.45 * heat.shape[0] + 1.6))
im = ax.imshow(heat.values, aspect="auto", cmap=LinearSegmentedColormap.from_list("seqblue", SEQ_BLUES))
ax.set_yticks(range(heat.shape[0])); ax.set_yticklabels(heat.index, color=INK)
ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels(heat.columns, fontsize=7)
ax.set_xlabel("Layer")
ax.set_title("Where does merging hurt?\nFunctional distortion of the K-Merge average on real inputs, per layer "
             "(darker = more conflict)", loc="left", color=INK, fontsize=10)
cb = plt.colorbar(im, ax=ax, fraction=0.025); cb.outline.set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "fig_layer_conflict.png"), dpi=150); plt.show()

d_km = long[long.method == "kmerge"]
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, col, lab in ((axs[0], "cos", "cosine similarity of the pair (K-Merge's rule)"),
                     (axs[1], "dgen", "predicted distortion on generated inputs (GRAM-Select)")):
    ax.scatter(d_km[col], d_km.damage, s=40, color="#2a78d6", edgecolor=SURF, linewidth=1, zorder=2)
    ax.grid(color=GRID, lw=0.8, zorder=0)
    rho = _spearman(-d_km.cos if col == "cos" else d_km[col], d_km.damage)
    ttl = "rho(-cosine, damage)" if col == "cos" else "rho(distortion, damage)"
    ax.set_xlabel(lab); ax.set_title(f"{ttl} = {rho:+.2f}", loc="left", color=INK, fontsize=10)
axs[0].set_ylabel("damage = 1 - normalised EM")
fig.suptitle("Which score predicts how much a K-Merge merge hurts a language?", x=0.01, ha="left", color=INK, fontsize=11)
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "fig_predictors.png"), dpi=150); plt.show()


def fig_merge_vs_share(pw_, path):
    """Dumbbell per language pair: K-Merge's average vs K-Share (worst-language normalised EM, mean over seeds)."""
    if not {"kmerge", "kshare"} <= set(pw_.method):
        return
    dd = pw_[pw_.method.isin(["kmerge", "kshare"])].groupby(["pair", "method"]).worst_nem.mean().unstack("method").dropna()
    if dd.empty:
        return
    cosp = pw_[pw_.method == "kmerge"].groupby("pair").cos.mean()
    dd = dd.loc[dd.kmerge.sort_values(ascending=False).index]          # the pair K-Merge handles worst ends up on top
    fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(dd) + 1.9))
    yy = np.arange(len(dd))
    for y_, (_, r_) in zip(yy, dd.iterrows()):
        ax.plot([r_.kmerge, r_.kshare], [y_, y_], color=AXIS, lw=2, zorder=1, solid_capstyle="round")
        ax.text(max(r_.kmerge, r_.kshare) + 0.015, y_, f"{r_.kshare - r_.kmerge:+.2f}", va="center", fontsize=8, color=INK2)
    ax.scatter(dd.kmerge, yy, s=70, color=GROUP_COLORS["prior"], edgecolor=SURF, linewidth=1.5, zorder=3,
               label="K-Merge: average the two adapters")
    ax.scatter(dd.kshare, yy, s=70, color=GROUP_COLORS["K-Share"], edgecolor=SURF, linewidth=1.5, zorder=3,
               label="K-Share: shared basis + per-task core")
    ax.set_yticks(yy)
    ax.set_yticklabels([f"{p_}  (cos {cosp.get(p_, np.nan):.2f})" for p_ in dd.index], color=INK)
    lo_ = float(np.nanmin(dd.values))
    ax.set_xlim(max(0.0, lo_ - 0.08), max(1.0, float(np.nanmax(dd.values))) + 0.09)
    ax.set_ylim(-0.6, len(dd) - 0.4)
    ax.grid(axis="x", color=GRID, lw=0.8, zorder=0)
    ax.set_xlabel("Worst-language normalised EM in the shared slot (1.0 = no loss); label = K-Share minus K-Merge")
    ax.set_title("Merging vs sharing, per language pair", loc="left", color=INK, fontsize=11)
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.2), ncol=2, fontsize=8)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()


fig_merge_vs_share(pw, os.path.join(RUN_DIR, "fig_merge_vs_share.png"))


def fig_capacity(tab, path):
    """Normalised EM vs number of tasks sharing one slot (lines; legend + de-overlapped direct labels)."""
    if tab is None or tab.empty:
        return
    av = tab["avg_normEM"]
    series = [(m_, lab_, col_) for m_, lab_, col_ in
              (("kmerge", "K-Merge average", GROUP_COLORS["prior"]),
               ("gramU", "GRAM-U (32 unlabeled inputs)", GROUP_COLORS["GRAM"]),
               ("split", "split (r/M directions per task)", GROUP_COLORS["K-Share"]),
               ("kshare", "K-Share", GROUP_COLORS["K-Share"])) if m_ in av.index]
    xs = [int(x_) for x_ in av.columns]
    fig, ax = plt.subplots(figsize=(7.2, 3.9))
    ends = []
    for m_, lab_, col_ in series:
        ys = av.loc[m_].to_numpy(dtype=float)
        dashed = m_ == "split"                          # same family as K-Share: same hue, dashed + hollow markers
        ax.plot(xs, ys, color=col_, lw=2, ls=(0, (4, 2.5)) if dashed else "-", marker="o", ms=7, mec=col_ if dashed else SURF,
                mfc=SURF if dashed else col_, mew=1.5, zorder=3, label=lab_)
        ends.append([float(ys[-1]), lab_.split(" (")[0]])
    lo_, hi_ = float(np.nanmin(av.loc[[s_[0] for s_ in series]].values)), float(np.nanmax(av.loc[[s_[0] for s_ in series]].values))
    ax.set_ylim(lo_ - 0.04, max(1.02, hi_ + 0.02))
    gap = 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0])
    ends.sort(key=lambda e_: e_[0])
    for i_ in range(1, len(ends)):                     # push labels apart so they never collide
        ends[i_][0] = max(ends[i_][0], ends[i_ - 1][0] + gap)
    for y_, txt_ in ends:
        ax.text(xs[-1] + 0.1, y_, txt_, va="center", fontsize=8, color=INK2)
    ax.set_xticks(xs); ax.set_xticklabels([f"{x_} tasks" for x_ in xs])
    ax.set_xlim(xs[0] - 0.2, xs[-1] + 0.85)
    ax.grid(axis="y", color=GRID, lw=0.8, zorder=0)
    ax.set_ylabel("normalised EM (1.0 = no loss)")
    ax.set_title(f"How many tasks can share one rank-{R} slot?", loc="left", color=INK, fontsize=11)
    ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, -0.13), ncol=2, fontsize=8, handlelength=4.2)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()


fig_capacity(cap_tab, os.path.join(RUN_DIR, "fig_capacity.png"))

# ---------------- summary ----------------
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
lines = ["=" * 100, "SEND THIS BACK", "=" * 100,
         f"mode={RUN_MODE} model={MODEL_NAME} seeds={SEEDS} pairs={len(PAIRS)} methods={len(METHOD_LIST)} "
         f"train/val/test={N_TRAIN}/{N_VAL}/{N_TEST} epochs={MODE['epochs']} lr={MODE['lr']} shared_init={SHARED_INIT} "
         f"layers={N_LAYERS} runtime={(time.time() - T0) / 60:.0f} min",
         "", "[1] ADAPTER QUALITY (EM on test; base = no adapter)",
         REF.groupby("lang")[["base_em", "solo_em", "solo_f1", "norm_dW"]].mean().round(3).to_string(),
         "", "[2] LEADERBOARD (normalised EM = merged / own adapter; 1.0 = no loss)",
         board.round(3).to_string(),
         "", "[3] FAILURE MODES (copy_underfix=F1 dilution, leak=F2 interference, imbalance=F3, norm_dominance_rho=F4)",
         fail.round(3).to_string(),
         "", "[4] WORST-LANGUAGE NORMALISED EM PER PAIR",
         pw.pivot_table(index="method", columns="pair", values="worst_nem").loc[board.index].round(2).to_string(),
         "", "[5] WHICH SCORE PREDICTS DAMAGE? Spearman rho with (1 - normalised EM); higher = better (F5, F6)",
         pred.round(3).to_string(index=False),
         "", "[6] LAYERS WITH MOST FUNCTIONAL CONFLICT (top 5 per pair)",
         "\n".join(f"  {p}: {v}" for p, v in top_layers.items())]
if seq_tab is not None:
    lines += ["", "[7] ONLINE K-MERGE (all arrival orders): policy x merge rule",
              seq_tab.round(3).to_string(index=False)]
# ---- [9] merge vs share: the head-to-head that matters for the paper ----
merge_only = board[board.group != "K-Share"]
best_merge = merge_only.avg_normEM.idxmax() if len(merge_only) else None
vs_rows = []
for m_, lab_ in (("kmerge", "K-Merge (average)"), (best_merge, f"best merge ({best_merge})"),
                 ("split", "split: own top directions (TSV-C idea)"), ("kshare", "K-Share (new)"),
                 ("kshare_eq", "K-Share, storage-exact (new)")):
    if m_ in board.index:
        r_ = board.loc[m_]
        vs_rows.append({"method": lab_, "avg_normEM": r_.avg_normEM, "worst_normEM": r_.worst_normEM,
                        "functional_damage": float(pw[pw.method == m_][["dact_a", "dact_b"]].mean().mean()),
                        "leak": fail.loc[m_, "leak"], "copy_underfix": fail.loc[m_, "copy_underfix"]})
lines += ["", "[9] MERGE vs SHARE (pairwise; functional_damage = output change on real inputs, lower is better)",
          pd.DataFrame(vs_rows).round(3).to_string(index=False),
          f"    storage: K-Share cores cost {100 * core_overhead_fraction():.2f}% of one adapter per task; "
          f"kshare_eq fits inside one slot exactly"]
if seq_tab is not None:
    keep = [("kmerge", "kmerge"), ("gramG", "global_dist"), ("split", "global_loss"),
            ("kshare", "kmerge"), ("kshare", "global_loss"), ("kshare", "layer_loss")]
    sv = seq_tab[[(r_.rule, r_.policy) in keep for r_ in seq_tab.itertuples()]]
    lines += ["", "[10] ONLINE: merging vs sharing (all arrival orders)", sv.round(3).to_string(index=False)]
lines += ["", "[8] VAL-SELECTED HYPER-PARAMETERS (most common per method)",
          pw.groupby("method").hp.agg(lambda s: s.value_counts().index[0]).to_string()]


# ---- [11] statistical significance: paired bootstrap over test sentences (+ McNemar), Holm-corrected ----
def significance(B=4000, ref="kmerge", rs=0):
    """Every method is scored on the SAME test sentences, so differences are tested pairwise: each bootstrap draw
    resamples the sentences of every (seed, language) once and re-scores all methods on that draw."""
    P = pd.read_csv(PRED_PATH, keep_default_na=False).drop_duplicates(["key", "lang", "src", "tgt"], keep="last")
    P["ex"] = list(zip(P.src, P.tgt))
    P["em"] = (P.pred.map(_norm) == P.tgt.map(_norm)).astype(float)
    chosen = {(int(r_.seed), frozenset(str(r_.pair).split("-")), r_.method): str(r_.hp) for r_ in pw.itertuples()}
    solo = {}
    for key, g in P[P.key.str.startswith("solo|")].groupby("key"):
        _, s_, l_ = key.split("|")
        if int(s_) in SEEDS:
            solo[(int(s_), l_)] = dict(zip(g.ex, g.em))
    cells = []                                   # (method, seed, pair, lang, em vector aligned to the solo sentences)
    for key, g in P.groupby("key", sort=False):
        p_ = key.split("|")
        if p_[0] == "m" and len(p_) == 5:
            seed_, meth_, hp_, tree_ = int(p_[1]), p_[2], p_[3], p_[4]
        elif p_[0] == "s" and len(p_) == 4:
            seed_, meth_, hp_, tree_ = int(p_[1]), p_[2], "-", p_[3].split("@")[0]
        else:
            continue
        mm_ = re.fullmatch(r"\((\w+)\+(\w+)\)", tree_)
        if not mm_ or seed_ not in SEEDS or chosen.get((seed_, frozenset(mm_.groups()), meth_)) != hp_:
            continue                             # only the configuration that was selected on validation
        for lang_, gl in g.groupby("lang"):
            ref_src = solo.get((seed_, lang_))
            if ref_src is None:
                continue
            got = dict(zip(gl.ex, gl.em))
            if set(got) != set(ref_src):
                continue
            order_ = list(ref_src)
            cells.append((meth_, seed_, "-".join(sorted(mm_.groups())), lang_,
                          np.array([got[s] for s in order_]), np.array([ref_src[s] for s in order_])))
    if not cells:
        return None, None
    rng = np.random.default_rng(rs)
    idx = {k: rng.integers(0, len(v), (B, len(v))) for k, v in solo.items()}
    R_ = {}
    for meth_ in sorted({c[0] for c in cells}):
        cc = [c for c in cells if c[0] == meth_]
        pts, bts, groups = [], [], defaultdict(list)
        for j, (_, s_, pr_, l_, em_, so_) in enumerate(cc):
            ix = idx[(s_, l_)]
            mb, sb = em_[ix].mean(1), so_[ix].mean(1)
            bts.append(np.where(sb >= 0.2, mb / np.maximum(sb, 1e-9), mb))
            pts.append(nz(em_.mean(), so_.mean()))
            groups[(s_, pr_)].append(j)
        pts, bts = np.array(pts), np.stack(bts)
        R_[meth_] = dict(avg=pts.mean(), avg_b=bts.mean(0),
                         worst=np.mean([pts[g_].min() for g_ in groups.values()]),
                         worst_b=np.mean([bts[g_].min(0) for g_ in groups.values()], axis=0),
                         em={(c[1], c[2], c[3]): c[4] for c in cc})

    def pval(x):
        return float(min(1.0, 2 * min((x <= 0).mean(), (x >= 0).mean())))

    def mcn(a_, b_):
        w_ = l_ = 0
        for k_, v_ in R_[a_]["em"].items():
            if k_ in R_[b_]["em"]:
                w_ += int(((v_ == 1) & (R_[b_]["em"][k_] == 0)).sum()); l_ += int(((v_ == 0) & (R_[b_]["em"][k_] == 1)).sum())
        p_ = float(sum(math.comb(w_ + l_, i) for i in range(min(w_, l_) + 1)) * 2 / 2 ** (w_ + l_)) if w_ + l_ else 1.0
        return w_, l_, min(1.0, p_)

    rows_ = []
    for m_, v_ in R_.items():
        lo_, hi_ = np.percentile(v_["avg_b"], [2.5, 97.5]); wlo_, whi_ = np.percentile(v_["worst_b"], [2.5, 97.5])
        row_ = {"method": m_, "avg_normEM": v_["avg"], "avg_95CI": f"[{lo_:.3f}, {hi_:.3f}]",
                "worst_normEM": v_["worst"], "worst_95CI": f"[{wlo_:.3f}, {whi_:.3f}]"}
        if ref in R_ and m_ != ref:
            w_, l_, p_ = mcn(m_, ref)
            row_.update({"d_avg_vs_kmerge": v_["avg"] - R_[ref]["avg"], "p_avg": pval(v_["avg_b"] - R_[ref]["avg_b"]),
                         "d_worst_vs_kmerge": v_["worst"] - R_[ref]["worst"],
                         "p_worst": pval(v_["worst_b"] - R_[ref]["worst_b"]),
                         "sentences_won/lost": f"+{w_}/-{l_}", "mcnemar_p": p_})
        rows_.append(row_)
    sig_ = pd.DataFrame(rows_).sort_values(["avg_normEM", "worst_normEM"], ascending=False).reset_index(drop=True)
    msk_ = sig_.method != ref
    for col_ in ("p_avg", "p_worst", "mcnemar_p"):
        if col_ in sig_:
            ps_ = sig_.loc[msk_, col_].fillna(1).to_numpy(); o_ = np.argsort(ps_); adj_ = np.empty(len(ps_)); run_ = 0.0
            for rank_, i_ in enumerate(o_):
                run_ = max(run_, (len(ps_) - rank_) * ps_[i_]); adj_[i_] = min(1.0, run_)
            sig_.loc[msk_, col_ + "_holm"] = adj_
    merges_ = [m_ for m_ in R_ if METHODS.get(m_, {}).get("group") is not None and m_ in board.index
               and board.loc[m_, "group"] != "K-Share"]
    best_ = max(merges_, key=lambda m_: R_[m_]["avg"]) if merges_ else None
    con_ = []
    for a_, b_ in (("kshare", "kmerge"), ("kshare", best_), ("kshare", "split"), ("kshare_eq", "kmerge"),
                   ("kshare_eq", best_), ("gramG", "kmerge"), ("gramU", "kmerge"), ("gramG", "regmeanG_cf"),
                   ("gramU", "regmean_cf"), ("gramG", "gramU"), (best_, "kmerge")):
        if a_ in R_ and b_ in R_ and a_ != b_:
            da_, dw_ = R_[a_]["avg_b"] - R_[b_]["avg_b"], R_[a_]["worst_b"] - R_[b_]["worst_b"]
            w_, l_, p_ = mcn(a_, b_)
            con_.append({"contrast": f"{a_} - {b_}", "d_avg_normEM": R_[a_]["avg"] - R_[b_]["avg"],
                         "avg_95CI": f"[{np.percentile(da_, 2.5):+.3f}, {np.percentile(da_, 97.5):+.3f}]", "p_avg": pval(da_),
                         "d_worst_normEM": R_[a_]["worst"] - R_[b_]["worst"],
                         "worst_95CI": f"[{np.percentile(dw_, 2.5):+.3f}, {np.percentile(dw_, 97.5):+.3f}]",
                         "p_worst": pval(dw_), "sentences_won/lost": f"+{w_}/-{l_}", "mcnemar_p": p_})
    return sig_, pd.DataFrame(con_)


sig, contrasts = significance()
if sig is not None:
    sig.to_csv(os.path.join(RUN_DIR, "significance.csv"), index=False)
    contrasts.to_csv(os.path.join(RUN_DIR, "contrasts.csv"), index=False)
    show_ = [c_ for c_ in ("method", "avg_normEM", "avg_95CI", "worst_normEM", "worst_95CI", "d_avg_vs_kmerge",
                           "p_avg_holm", "p_worst_holm", "sentences_won/lost", "mcnemar_p_holm") if c_ in sig]
    lines += ["", "[11] SIGNIFICANCE (paired bootstrap over test sentences, 4000 draws; p vs K-Merge, Holm-corrected)",
              sig[show_].round(4).to_string(index=False),
              "", "[12] KEY CONTRASTS (difference in normalised EM, 95% CI, two-sided p; McNemar on exact matches)",
              contrasts.round(4).to_string(index=False)]
if cap_tab is not None:
    lines += ["", f"[13] SLOT CAPACITY: M tasks share ONE rank-{R} slot (M=2 from the pairwise benchmark; M=3,4 = every "
                  "subset of languages); normalised EM, avg and worst language per slot",
              cap_tab.round(3).to_string()]
    cap_tab.to_csv(os.path.join(RUN_DIR, "capacity_summary.csv"))
lines += ["=" * 100]
summary = "\n".join(lines)
print(summary)
with open(os.path.join(RUN_DIR, "RESULTS_SUMMARY.txt"), "w") as f:
    f.write(summary)
board.to_csv(os.path.join(RUN_DIR, "leaderboard.csv")); fail.to_csv(os.path.join(RUN_DIR, "failure_modes.csv"))
pred.to_csv(os.path.join(RUN_DIR, "predictors.csv"), index=False)
if seq_tab is not None:
    seq_tab.to_csv(os.path.join(RUN_DIR, "sequence_summary.csv"), index=False)

zpath = os.path.join(OUT_DIR, "results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(RUN_DIR)):
        if fn.endswith((".csv", ".png", ".txt")):
            z.write(os.path.join(RUN_DIR, fn), arcname=f"{CFG_TAG}_{RUN_MODE}/{fn}")
log(f"done. Download {zpath} (Output tab) and send it with the summary above.")

## How to read the results

* **[2] Leaderboard:** `avg_normEM` 1.0 means the merged adapter is as good as each language's own adapter. Compare `gramG` (data-free) and `gramU` (32 unlabeled inputs) with `kmerge` (what K-Merge does) and with the strongest prior method. The `data` column shows what each method needs. **Methods marked "labeled val" were tuned on labeled validation data**, an advantage GRAM does not get.
* **[3] Failure modes:** high `copy_underfix` = dilution (F1); high `leak` = language interference (F2); high `imbalance` = one language wins (F3); positive `norm_dominance_rho` = the bigger adapter wins (F4).
* **[5] Predictors:** if the generated-input distortion rho is clearly higher than the cosine rho, you have evidence that *similarity ≠ compatibility* and a data-free score that fixes it (F5). If the self-Gram rho is low or negative, that confirms F6.
* **[6] Layers:** on Qwen2.5-1.5B (28 layers) check whether the EN-FR conflict concentrates in layers 17-21, as in your Analysis 1-4.
* **[7] Online:** compare `global_dist` (GRAM-Select) with `kmerge` (K-Merge's policy): higher `worst_order_avg` and lower `order_spread_std` = more robust to arrival order. `oracle` uses labeled data and is only an upper bound.
* **[9] Merge vs share:** the key table. If `kshare` and `split` have much higher normalised EM and much lower functional damage than every merge, the paper's main claim holds: *in K-Merge's setting, do not merge, share*. `kshare` vs `split` shows whether a shared basis beats a simple split; `kshare_eq` shows the result holds at exactly equal storage.
* **[10] Online merging vs sharing:** `kshare / layer_loss` vs `kshare / global_loss` isolates the value of per-layer slot assignment (your layer-localised conflict finding).
* **[11] Significance:** 95% confidence intervals for every method and p-values against K-Merge, from a *paired* bootstrap over the test sentences (every method is re-scored on the same resampled sentences) and from McNemar's test on exact matches; both Holm-corrected for the number of methods. Report a difference as real only if the Holm-corrected p is below 0.05.
* **[12] Key contrasts:** the comparisons the paper's claims rest on (K-Share vs K-Merge, vs the best merge, vs the split baseline; GRAM vs RegMean), each with its own confidence interval.
* **[13] Slot capacity:** 2, 3 and 4 tasks forced into one rank-8 slot. This is where a shared basis should separate from `split` (which leaves each task only r/M directions) and from merging (one adapter for M tasks). `fig_capacity.png` plots it; `fig_merge_vs_share.png` shows K-Merge vs K-Share on every pair.